# From access-period experiments to spatial crop-mixture epidemics

## Bayesian parameter inference, branching-process invasion risk, deterministic mean field, and finite spatial Gillespie dynamics

This notebook is a scientific reference implementation for the spatial extension of cassava varietal-mixture epidemiology.

It makes explicit that four mathematical/statistical layers are involved:

\[
\boxed{
\text{Access-period data}
\longrightarrow
\text{Bayesian inference in EpiPvr}
\longrightarrow
p(\Theta\mid D)
}
\]

followed by complementary epidemic representations:

\[
\boxed{
\begin{array}{ccc}
\text{EpiPvr branching process}
&
\text{PLOS mean-field ODE}
&
\text{spatial CTMC / Gillespie}
\\[1mm]
\text{early invasion risk}
&
\text{population-average dynamics}
&
\text{finite stochastic spatial dynamics}
\end{array}
}
\]

These are **not competing algorithms for the same task**.

- Bayesian analysis estimates uncertain biological transmission parameters from laboratory data.
- The EpiPvr branching process uses those parameters to infer stochastic establishment/extinction risk after rare introduction.
- The PLOS ODE model describes deterministic population-average epidemic dynamics.
- The spatial Gillespie model explicitly simulates finite stochastic epidemics and vector movement on a planted field.

The spatial work shares the same biological parameter lineage but does not assume that the published EpiPvr branching process, the PLOS ODE and the spatial CTMC are algebraically identical. Their assumptions differ and are stated explicitly.

### Primary sources

**Tankam Chedjou, I., Donnelly, R. & Gilligan, C. A. (2025).**  
*Optimizing crop varietal mixtures for viral disease management: A case study on cassava virus epidemics.*  
PLOS Computational Biology 21(9): e1012842.  
https://doi.org/10.1371/journal.pcbi.1012842

**Donnelly, R., Tankam Chedjou, I. & Gilligan, C. A. (2026).**  
*Plant pathogen profiling with the EpiPvr package.*  
Methods in Ecology and Evolution 17: 837–849.  
https://doi.org/10.1111/2041-210x.70219

EpiPvr R package: https://cran.r-project.org/package=EpiPvr

PLOS mixture-model implementation: https://github.com/israeltankam/mixture-simulator

---

## Scope of the current notebook

The present paper remains restricted to:

- a square \(N\times N\) field;
- \(N=10\) by default as a computational prototype;
- two cassava varieties (`SUSC`, `RES`);
- exact 50:50 composition for arrangement comparisons;
- the canonical planting designs already defined in the draft;
- CBSD/CBSI semi-persistent transmission;
- fixed vector burden \(m\) per plant;
- no vector attractiveness or host-choice preference;
- one growing season;
- no roguing in the principal arrangement experiment.

The future `Cropmix` package may generalise these restrictions. That software generality is not silently imported into the present paper.

---

## Principal methodological corrections retained here

1. Exact conservative pairwise vector exchange.
2. Symmetric, row-stochastic finite-field movement kernel.
3. Explicit **dispersion** contribution in the Gillespie intensity.
4. No plant attractiveness term.
5. Kernel-scale calibration from complete epidemic trajectories, not final yield alone.
6. One common kernel scale across epidemic-pressure contexts.
7. Explicit identifiability diagnostics.
8. Robustness of planting conclusions across the acceptable kernel-scale region.
9. Executable reproducibility checks.
10. Explicit separation of **parameter uncertainty** from **process stochasticity**.
11. Explicit explanation of how EpiPvr Bayesian inference, EpiPvr branching-process risk, PLOS mean-field dynamics and Cropmix-style Gillespie simulation fit together.



# 1. Model lineage, notation, and source audit

The executable PLOS implementation uses the following convention, which is adopted throughout this notebook:

- \(\alpha_v\): virus **acquisition** rate by a virus-free vector feeding on infectious cultivar \(v\);
- \(\beta_v\): virus **inoculation** rate for cultivar \(v\) exposed to a viruliferous vector;
- \(\gamma_v\): latent-to-infectious progression rate of a plant;
- \(\rho_v\): roguing rate;
- \(\sigma\): individual vector dispersal rate;
- \(\omega\): vector mortality rate;
- \(r\): loss of vector infectivity.

The public prose of the PLOS article contains a notation reversal in one paragraph describing \(\alpha\) and \(\beta\). The repository implementation and the application settings consistently use \(\alpha=\) acquisition and \(\beta=\) inoculation; those code-consistent biological meanings are used here.

The spatial exponential-kernel scale is denoted by \(a\), never by \(\alpha\):

\[
K_a(d)=e^{-d/a},\qquad a>0.
\]

Thus:

- \(\alpha_v\) is a biological acquisition rate;
- \(a\) is a spatial e-folding distance;
- \(\bar d(a)\) is the finite-field mean movement distance implied by the normalised kernel.

For a planting spacing of \(1\) m, one plant-spacing unit is numerically one metre. The notebook keeps the more general unit “plant spacing”.


In [ ]:
# Core scientific Python imports and reproducibility configuration
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from collections import deque
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from numba import njit, prange, set_num_threads, get_num_threads

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------
MASTER_SEED = 20260810

# FAST_MODE is a software-validation / exploratory mode, not publication evidence.
FAST_MODE = True

# The 100-point scale scan is always retained.
# Only the number of Monte Carlo replicates changes with FAST_MODE.
N_CALIBRATION_RUNS = 2 if FAST_MODE else 250
N_PATTERN_RUNS = 10 if FAST_MODE else 2000
N_KERNEL_ROBUSTNESS_RUNS = 5 if FAST_MODE else 1000
N_BOOTSTRAP = 50 if FAST_MODE else 1000
N_METRIC_BOOTSTRAP = 50 if FAST_MODE else 5000

# Observation grid used for trajectory matching and plotting.
# 5-day spacing over a 360-day season.
N_OBSERVATION_TIMES = 73

set_num_threads(min(6, get_num_threads()))

print(f"Python: {sys.version.split()[0]}")
print(f"Numba threads: {get_num_threads()}")
print(f"FAST_MODE={FAST_MODE}")
print(f"Calibration runs per (a, context): {N_CALIBRATION_RUNS}")
print(f"Pattern runs: {N_PATTERN_RUNS}")

# 2. Modelling hierarchy: inference and epidemic dynamics answer different questions

Let \(D\) denote access-period experimental data and let \(\Theta\) denote biological transmission parameters.

For an SPT virus such as CBSI/CBSD,

\[
\Theta=(\alpha,\beta,\mu),
\]

where acquisition, inoculation and vector clearance are estimated from access-period data. PT viruses additionally require vector latent progression.

The inferential pipeline is

\[
D
\overset{\text{Bayes / Stan}}{\longrightarrow}
p(\Theta\mid D),
\]

then, conditional on \(\Theta\),

\[
\Theta
\longrightarrow
\begin{cases}
\text{branching-process invasion calculation},\\
\text{deterministic mean-field dynamics},\\
\text{finite spatial stochastic dynamics}.
\end{cases}
\]

The same rates can legitimately appear in models solved by different mathematical techniques because those techniques answer different questions.


## 2.1 EpiPvr Step A: Bayesian inference from access-period experiments

A generic access-period sub-assay observes \(Y_j\) infected test plants among \(n_j\) replicates after controlled feeding durations.

The mechanistic access-period model implies an infection probability \(p_j(\Theta)\), giving an observation model of the form

\[
Y_j\mid\Theta
\sim
\operatorname{Binomial}\!\left(n_j,p_j(\Theta)\right).
\]

With prior \(p(\Theta)\),

\[
\boxed{
p(\Theta\mid D)
\propto
p(D\mid\Theta)p(\Theta).
}
\]

EpiPvr uses Stan MCMC to sample this posterior.

For SPT transmission, `estimate_virus_parameters_SPT()` estimates posterior distributions for acquisition, inoculation and vector-clearance rates. For PT transmission, `estimate_virus_parameters_PT()` additionally handles vector latent progression.

### The joint posterior is the important object

The complete result is a set of coherent draws

\[
\Theta^{(b)}
=
(\alpha^{(b)},\beta^{(b)},\mu^{(b)}),
\qquad b=1,\ldots,B.
\]

Posterior correlation must be preserved. Independently sampling marginal posteriors generally does not reproduce the EpiPvr posterior.

### Units

When an EpiPvr posterior is expressed per hour and the field model uses days,

\[
\alpha_{\mathrm{day}}=24\alpha_{\mathrm{hour}},
\]

with the same conversion for the other rate parameters.


## 2.2 EpiPvr Step B: multitype branching process for early field invasion

EpiPvr next combines transmission-rate estimates with local field parameters to infer whether a rare introduction establishes or fades out stochastically.

When infection is rare:

- susceptible hosts are effectively undepleted;
- lineages interact weakly;
- descendants can be approximated as independent;
- extinction versus establishment is the central question.

This is the natural regime for a multitype branching process.

The EpiPvr field construction contains active types broadly of the form

\[
I_j,\qquad E_j,\qquad S_j,
\]

where \(j\) is the number of virus-bearing vectors associated with the plant. It represents acquisition, inoculation, plant latent progression, vector mortality, clearance, dispersal, plant removal/roguing and harvest. Dispersal of a virus-bearing vector can generate a new inoculum lineage.

If \(q_k\) is extinction probability from inoculum type \(k\),

\[
\mathbf q=\mathbf G(\mathbf q),
\]

and

\[
\boxed{
P_{\mathrm{epidemic},k}=1-q_k.
}
\]

EpiPvr exposes this layer through `calculate_epidemic_probability()`.

EpiPvr Step B treats within-field movement **implicitly in space**: it has a dispersal rate but no explicit plant coordinates, distance kernel or planting design.


## 2.3 The EpiPvr branching process is related to, but not automatically identical to, the spatial Gillespie model

A branching process is the natural rare-invasion approximation to a suitable stochastic epidemic process. However, it is too strong to assert automatically that

\[
\text{published EpiPvr branching process}
=
\text{exact low-prevalence linearisation of this spatial CTMC}.
\]

The constructions differ:

| Feature | EpiPvr branching process | Present spatial notebook |
|---|---|---|
| space | implicit | explicit coordinates |
| movement | dispersal rate | distance-weighted conservative swaps |
| geometry | implicit | \(N\times N\) |
| vector burden | fixed | fixed |
| susceptible depletion | neglected in rare-lineage approximation | explicit |
| removal/harvest | part of invasion process | no roguing in primary experiment; terminal harvest |
| PT vector latency | supported | absent in current SPT engine |
| planting design | absent | central |

They share a biological parameter lineage, but numerical equality of establishment probabilities requires a deliberately harmonised validation scenario.


## 2.4 Why Gillespie is the appropriate engine for the spatial mixture problem

With explicit planting design, the epidemic is a finite continuous-time Markov chain.

If state-changing events at state \(X\) have hazards

\[
\lambda_1(X),\ldots,\lambda_K(X),
\]

then

\[
\Lambda(X)=\sum_k\lambda_k(X),
\]

\[
\tau\sim\operatorname{Exponential}(\Lambda(X)),
\]

and event \(k\) is selected with probability

\[
P(k\mid X)=\frac{\lambda_k(X)}{\Lambda(X)}.
\]

Gillespie simulation gives full random trajectories \(X(t)\), from which we obtain seasonal incidence, vector prevalence, spatial maps, finite-field effects, planting-arrangement effects and final yield distributions.

Moving from EpiPvr to Gillespie therefore changes the dynamical question, not the biological meaning of the transmission rates.


## 2.5 Parameter uncertainty and epidemic stochasticity are different uncertainties

Parameter uncertainty:

\[
\Theta\sim p(\Theta\mid D).
\]

This is handled by EpiPvr.

Process stochasticity:

\[
X(t)\sim p_{\mathrm{CTMC}}(X\mid z,\Theta),
\]

where \(z\) is planting design.

The full posterior predictive distribution is

\[
\boxed{
p(\mathcal O\mid D,z)
=
\int
p_{\mathrm{CTMC}}(\mathcal O\mid z,\Theta)
p_{\mathrm{EpiPvr}}(\Theta\mid D)\,d\Theta.
}
\]

Nested Monte Carlo uses posterior draws \(\Theta^{(b)}\) and, within each draw, repeated Gillespie trajectories \(X^{(b,r)}\).

The current paper conditions principally on published point estimates to isolate spatial-design effects. Thus its Monte Carlo variability is process variability conditional on \(\widehat\Theta\), not total posterior predictive uncertainty.


## 2.6 Notation crosswalk

| Biological quantity | EpiPvr | PLOS mixture | Spatial notebook |
|---|---:|---:|---:|
| acquisition | \(\alpha\) | \(\alpha\) | \(\alpha_v\) |
| inoculation | \(\beta\) | \(\beta\) | \(\beta_v\) |
| vector clearance | \(\mu\) | \(r\) | `VECTOR_CLEARANCE` |
| vector latent progression (PT) | \(\gamma\) | not explicit | absent in current SPT engine |
| plant latent progression | \(\nu\) | \(\gamma\) | \(\gamma_v\) |
| vector dispersal rate | \(\theta\) | \(\sigma\) | \(\sigma\) |
| vector mortality | \(b_f\) | \(\omega\) | \(\omega\) |
| roguing | \(r\) | \(\rho\) | \(\rho_v\) |
| spatial kernel scale | not explicit | not explicit | \(a\) |

The spatial kernel scale is always \(a\), never \(\alpha\).


In [ ]:
# Optional EpiPvr posterior interface.
# This notebook does not require R at runtime to reproduce the point-estimate study.

@dataclass(frozen=True)
class MixtureTransmissionDraw:
    """One coherent joint draw for the current two-variety SPT model."""
    susc_acquisition: float
    susc_inoculation: float
    res_acquisition: float
    res_inoculation: float
    vector_clearance: float


def load_joint_epipvr_mixture_posterior(csv_path, rate_unit="per_day"):
    """Load coherent joint posterior rows exported through the future Cropmix/R bridge."""
    df = pd.read_csv(csv_path)
    required = [
        "susc_acquisition",
        "susc_inoculation",
        "res_acquisition",
        "res_inoculation",
        "vector_clearance",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing posterior columns: {missing}")

    posterior = df[required].copy()

    if rate_unit == "per_hour":
        posterior[required] = 24.0 * posterior[required]
    elif rate_unit != "per_day":
        raise ValueError("rate_unit must be 'per_day' or 'per_hour'.")

    values = posterior.to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError("Posterior contains non-finite values.")
    if (values < 0).any():
        raise ValueError("Transmission rates cannot be negative.")

    return posterior


def posterior_draw_from_row(row):
    return MixtureTransmissionDraw(
        susc_acquisition=float(row["susc_acquisition"]),
        susc_inoculation=float(row["susc_inoculation"]),
        res_acquisition=float(row["res_acquisition"]),
        res_inoculation=float(row["res_inoculation"]),
        vector_clearance=float(row["vector_clearance"]),
    )


EPIPVR_POSTERIOR_CSV = None


# 3. CBSD parameterisation and point-estimate analysis

The case study concerns cassava brown streak disease / cassava brown streak ipomovirus (CBSI/CBSD), treated as a semi-persistently transmitted system.

The current spatial engine uses the SPT-compatible transmission set

\[
(\alpha_v,\beta_v,r)
\]

and does **not** contain a vector latent compartment.

A PT pathogen cannot be inserted into this CTMC merely by providing EpiPvr's vector latent-progression estimate. A PT Cropmix engine must introduce an exposed-vector state and the corresponding transitions.

The current case study uses the susceptible (`SUSC`) and resistant (`RES`) phytotypes from the PLOS CBSD analysis.

| Quantity | SUSC | RES |
|---|---:|---:|
| acquisition rate | \(15.31\ \mathrm{day}^{-1}\) | \(4.84\ \mathrm{day}^{-1}\) |
| inoculation rate | \(1.34\ \mathrm{day}^{-1}\) | \(0.42\ \mathrm{day}^{-1}\) |
| healthy yield | \(31\ \mathrm{t\,ha}^{-1}\) | \(25\ \mathrm{t\,ha}^{-1}\) |
| infected yield | \(3.1\ \mathrm{t\,ha}^{-1}\) | \(2.1\ \mathrm{t\,ha}^{-1}\) |

Shared CBSD quantities are

\[
\sigma=0.45\ \mathrm{day}^{-1},\qquad
\omega=0.19\ \mathrm{day}^{-1},\qquad
r=19.37\ \mathrm{day}^{-1},
\]

with plant latent progression

\[
\gamma=\frac1{30}\ \mathrm{day}^{-1}.
\]

The season length is \(T=360\) days and the principal arrangement experiment has no roguing, \(\rho=0\).

The PLOS model assumes one initially infectious plant and initially virus-free vectors.

### Inferential status

The principal analysis conditions on published point values:

\[
p(\mathcal O\mid\widehat\Theta,z).
\]

It does not yet integrate over

\[
p(\Theta\mid D).
\]

A later Cropmix uncertainty analysis should propagate coherent joint EpiPvr posterior draws.


In [ ]:
@dataclass(frozen=True)
class Cultivar:
    name: str
    acquisition: float          # alpha_v, day^-1
    inoculation: float           # beta_v, per viruliferous vector per day
    latency_progression: float   # gamma_v, day^-1
    roguing: float               # rho_v, day^-1
    yield_healthy: float         # t ha^-1
    yield_infected: float        # t ha^-1


SUSC = Cultivar(
    name="SUSC",
    acquisition=15.31,
    inoculation=1.34,
    latency_progression=1.0 / 30.0,
    roguing=0.0,
    yield_healthy=31.0,
    yield_infected=3.1,
)

RES = Cultivar(
    name="RES",
    acquisition=4.84,
    inoculation=0.42,
    latency_progression=1.0 / 30.0,
    roguing=0.0,
    yield_healthy=25.0,
    yield_infected=2.1,
)

SIGMA = 0.45
OMEGA = 0.19
VECTOR_CLEARANCE = 19.37
T_END = 360.0
PAPER_K = 10_000
PLANT_SPACING_M = 1.0

parameter_table = pd.DataFrame([
    ["SUSC acquisition", r"$\alpha_S$", SUSC.acquisition, "day^-1"],
    ["SUSC inoculation", r"$\beta_S$", SUSC.inoculation, "vector^-1 day^-1"],
    ["RES acquisition", r"$\alpha_R$", RES.acquisition, "day^-1"],
    ["RES inoculation", r"$\beta_R$", RES.inoculation, "vector^-1 day^-1"],
    ["Plant latent progression", r"$\gamma$", SUSC.latency_progression, "day^-1"],
    ["Vector dispersal", r"$\sigma$", SIGMA, "day^-1"],
    ["Vector mortality", r"$\omega$", OMEGA, "day^-1"],
    ["Loss of vector infectivity", r"$r$", VECTOR_CLEARANCE, "day^-1"],
    ["Season length", r"$T$", T_END, "days"],
    ["Published planting density", r"$K$", PAPER_K, "plants ha^-1"],
], columns=["quantity", "symbol", "value", "unit"])

parameter_table

POINT_TRANSMISSION_DRAW = MixtureTransmissionDraw(
    susc_acquisition=SUSC.acquisition,
    susc_inoculation=SUSC.inoculation,
    res_acquisition=RES.acquisition,
    res_inoculation=RES.inoculation,
    vector_clearance=VECTOR_CLEARANCE,
)



# 4. PLOS mean-field reference model

## 4.1 Published two-variety equations

For varieties \(A\) and \(B\), let

\[
l_A,\ i_A,\ l_B,\ i_B
\]

denote the fractions of the entire field occupied by latent and infectious plants, and let

\[
V^A,\ V^B
\]

be the numbers of viruliferous vectors classified according to the variety on which virus acquisition occurred.

The planted fractions are

\[
\theta_A=\theta,\qquad \theta_B=1-\theta.
\]

Define

\[
q=\omega+r,
\qquad
\psi=\frac{1}{\sigma+q},
\qquad
F=mK.
\]

The code-consistent PLOS equations are

\[
\frac{dl_A}{dt}
=
\frac{\sigma\psi}{K}
\beta_A(\theta-l_A-i_A)(V^A+V^B)
-\gamma_A l_A,
\]

\[
\frac{di_A}{dt}
=
\gamma_A l_A-\rho_A i_A,
\]

\[
\frac{dl_B}{dt}
=
\frac{\sigma\psi}{K}
\beta_B(1-\theta-l_B-i_B)(V^A+V^B)
-\gamma_B l_B,
\]

\[
\frac{di_B}{dt}
=
\gamma_B l_B-\rho_B i_B,
\]

\[
\frac{dV^A}{dt}
=
\alpha_A
\left[
i_AF-\psi\left\{\sigma i_A(V^A+V^B)+qV^A\right\}
\right]
-qV^A,
\]

\[
\frac{dV^B}{dt}
=
\alpha_B
\left[
i_BF-\psi\left\{\sigma i_B(V^A+V^B)+qV^B\right\}
\right]
-qV^B.
\]

For one introduced infectious plant,

\[
i_A(0)=\frac{\theta}{K},
\qquad
i_B(0)=\frac{1-\theta}{K},
\]

with all latent and viruliferous-vector states initially zero.

For \(I_0\) introduced infectious plants in the finite-grid consistency analysis, the deterministic ensemble analogue is

\[
i_A(0)=\theta\frac{I_0}{K},
\qquad
i_B(0)=(1-\theta)\frac{I_0}{K}.
\]

At harvest,

\[
Y(T)
=
Y_A^H\{\theta-i_A(T)\}+Y_A^I i_A(T)
+
Y_B^H\{1-\theta-i_B(T)\}+Y_B^I i_B(T).
\]

Latent plants are included with uninfected yield.


In [ ]:
def solve_plos_meanfield(
    theta: float,
    vector_burden: int,
    K: int,
    n_initial_infectious: int = 1,
    cultivar_a: Cultivar = SUSC,
    cultivar_b: Cultivar = RES,
    t_end: float = T_END,
    t_eval: np.ndarray | None = None,
):
    """Solve the PLOS reduced mean-field model with strict numerical tolerances."""
    if not 0.0 <= theta <= 1.0:
        raise ValueError("theta must lie in [0, 1].")
    if vector_burden <= 0:
        raise ValueError("vector_burden must be positive.")
    if not 0 <= n_initial_infectious <= K:
        raise ValueError("Invalid initial inoculum.")

    q = OMEGA + VECTOR_CLEARANCE
    psi = 1.0 / (SIGMA + q)
    F = float(vector_burden * K)

    def rhs(_t, y):
        l_a, i_a, l_b, i_b, v_a, v_b = y
        v_total = v_a + v_b

        s_a = theta - l_a - i_a
        s_b = (1.0 - theta) - l_b - i_b

        # Tiny negative values from floating-point integration are harmless,
        # but biological state fractions cannot be negative.
        s_a = max(s_a, 0.0)
        s_b = max(s_b, 0.0)

        dl_a = (
            (SIGMA * psi / K)
            * cultivar_a.inoculation
            * s_a
            * v_total
            - cultivar_a.latency_progression * l_a
        )
        di_a = cultivar_a.latency_progression * l_a - cultivar_a.roguing * i_a

        dl_b = (
            (SIGMA * psi / K)
            * cultivar_b.inoculation
            * s_b
            * v_total
            - cultivar_b.latency_progression * l_b
        )
        di_b = cultivar_b.latency_progression * l_b - cultivar_b.roguing * i_b

        dv_a = cultivar_a.acquisition * (
            i_a * F
            - psi * (SIGMA * i_a * v_total + q * v_a)
        ) - q * v_a

        dv_b = cultivar_b.acquisition * (
            i_b * F
            - psi * (SIGMA * i_b * v_total + q * v_b)
        ) - q * v_b

        return np.array([dl_a, di_a, dl_b, di_b, dv_a, dv_b], dtype=float)

    initial_fraction = n_initial_infectious / K
    y0 = np.array([
        0.0,
        theta * initial_fraction,
        0.0,
        (1.0 - theta) * initial_fraction,
        0.0,
        0.0,
    ])

    if t_eval is None:
        t_eval = np.linspace(0.0, t_end, N_OBSERVATION_TIMES)

    sol = solve_ivp(
        rhs,
        (0.0, t_end),
        y0,
        t_eval=t_eval,
        method="LSODA",
        rtol=1e-9,
        atol=1e-12,
    )
    if not sol.success:
        raise RuntimeError(sol.message)

    l_a, i_a, l_b, i_b, v_a, v_b = sol.y
    incidence = i_a + i_b
    vector_prevalence = (v_a + v_b) / F

    final_yield = (
        cultivar_a.yield_healthy * (theta - i_a[-1])
        + cultivar_a.yield_infected * i_a[-1]
        + cultivar_b.yield_healthy * ((1.0 - theta) - i_b[-1])
        + cultivar_b.yield_infected * i_b[-1]
    )

    return {
        "time": sol.t,
        "solution": sol.y,
        "incidence": incidence,
        "incidence_a": i_a,
        "incidence_b": i_b,
        "vector_prevalence": vector_prevalence,
        "yield": float(final_yield),
        "final_incidence": float(incidence[-1]),
    }


OBS_TIMES = np.linspace(0.0, T_END, N_OBSERVATION_TIMES)

# Published-scale reference and the finite N=10 reference.
meanfield_benchmark_rows = []
for K in (100, PAPER_K):
    for m in (1, 5, 10):
        mf = solve_plos_meanfield(
            theta=1.0,
            vector_burden=m,
            K=K,
            n_initial_infectious=1,
            t_eval=OBS_TIMES,
        )
        meanfield_benchmark_rows.append({
            "K": K,
            "vectors_per_plant": m,
            "initial_prevalence": 1.0 / K,
            "final_incidence": mf["final_incidence"],
            "final_yield_t_ha": mf["yield"],
        })

meanfield_benchmarks = pd.DataFrame(meanfield_benchmark_rows)
meanfield_benchmarks


# 5. Spatially explicit stochastic model

## 5.1 State variables

The prototype field contains

\[
M=N^2
\]

planting sites. Site \(i\) contains one plant and exactly \(m\) vectors.

Plant state:

\[
X_i(t)\in\{S,L,I\}.
\]

Cultivar identity:

\[
v_i\in\{A,B\}.
\]

Vector counts:

\[
H_i,\qquad V_i^A,\qquad V_i^B,
\]

with the strict invariant

\[
H_i+V_i^A+V_i^B=m
\]

for every site and every time.

The superscript on \(V_i^A\) or \(V_i^B\) records **where virus acquisition occurred**. It does not affect destination choice and does not encode attractiveness.

---

## 5.2 Local epidemiological events

For a plant of cultivar \(v_i\):

| process | propensity |
|---|---:|
| inoculation \(S\to L\) | \(\beta_{v_i}(V_i^A+V_i^B)\) |
| end of plant latency \(L\to I\) | \(\gamma_{v_i}\) |
| roguing \(I\to S\) | \(\rho_{v_i}\) |
| acquisition \(H\to V^{v_i}\) on an infectious plant | \(\alpha_{v_i}H_i\) |
| viruliferous-vector mortality and immediate replacement | \(\omega(V_i^A+V_i^B)\) |
| loss of infectivity | \(r(V_i^A+V_i^B)\) |

Healthy-vector mortality followed by immediate healthy replacement is epidemiologically state-null and does not need to be simulated.

### Why the local inoculation rate is not multiplied again by \(\sigma\psi\)

The PLOS ODE contains \(\sigma\psi\) inside its **population-averaged** inoculation term. In the spatial process, vector residence and movement are simulated explicitly. Multiplying the local feeding/inoculation hazard by the same movement factor would count the dispersal mechanism twice.

This spatial CTMC should therefore be understood as an event-level spatialisation of the same biological processes, **not as an algebraically guaranteed microscopic derivation of every PLOS ODE term**. That distinction is precisely why a mean-field-consistency analysis is required. If no value of \(a\) gives satisfactory simultaneous agreement in plant and vector trajectories, the discrepancy is structural and must not be hidden by forcing a kernel-scale estimate.



# 6. Exact conservative dispersal by pairwise exchange

## 6.1 Raw exponential distance weights

For plant coordinates \(x_i\) and \(x_j\),

\[
d_{ij}=\|x_i-x_j\|,
\qquad
W_{ij}(a)=
\begin{cases}
e^{-d_{ij}/a}, & i\ne j,\\
0, & i=j.
\end{cases}
\]

No attractiveness term is used.

---

## 6.2 Why ordinary row normalisation is not sufficient

On an open finite field, an edge plant has a different sum of raw kernel weights from an interior plant. Therefore

\[
\widetilde P_{ij}
=
\frac{W_{ij}}{\sum_kW_{ik}}
\]

is row-stochastic but generally

\[
\widetilde P_{ij}\ne\widetilde P_{ji}.
\]

That asymmetry is incompatible with a reciprocal pair-swap process if every tagged vector is required to move at exactly the same rate \(\sigma\).

We therefore balance the symmetric matrix \(W\) into a matrix \(P\) satisfying

\[
P_{ij}=P_{ji},\qquad
P_{ii}=0,\qquad
\sum_{j}P_{ij}=1.
\]

A symmetric Sinkhorn scaling is used numerically. For the complete positive off-diagonal exponential graph used here, the scaling is well behaved.

---

## 6.3 Physical pair generator

For each unordered pair \(\{i,j\}\), \(i<j\), define the physical swap-event rate

\[
\boxed{
\lambda_{ij}=m\sigma P_{ij}.
}
\]

When the event occurs:

1. select one of the \(m\) vectors at \(i\) uniformly;
2. select one of the \(m\) vectors at \(j\) uniformly;
3. exchange the selected vectors.

Because \(P\) is symmetric and row-stochastic,

\[
\sum_{j\ne i}\lambda_{ij}\frac{1}{m}
=
\sigma\sum_{j\ne i}P_{ij}
=
\boxed{\sigma}.
\]

Thus **every tagged vector moves at the published individual dispersal rate \(\sigma\)**.

The total physical rate of pair events is

\[
\Lambda_{\mathrm{pair}}
=
\sum_{i<j}m\sigma P_{ij}
=
\frac{Mm\sigma}{2}.
\]

Each pair event moves two vectors, so the total rate of individual vector movements is

\[
2\Lambda_{\mathrm{pair}}=Mm\sigma,
\]

again giving \(\sigma\) movements per vector per unit time.

This is the rigorous origin of the “factor of two”. If one defines a per-vector *pair-initiation* bookkeeping rate \(\nu\), then \(\nu=\sigma/2\); however, the simulator does **not** use that heuristic. It uses the pair generator above.

---

## 6.4 State-changing dispersal intensity

Most physical swaps are epidemiologically invisible, for example healthy \(\leftrightarrow\) healthy. Such null events may be integrated out without changing the CTMC of the tracked epidemic state.

Let

\[
V_i=V_i^A+V_i^B,\qquad H_i=m-V_i.
\]

The total intensity of dispersal events that change at least one tracked vector-count component is

\[
\boxed{
\Lambda_{\mathrm{disp}}(t)
=
\frac{\sigma}{m}
\sum_{i<j}P_{ij}
\left[
H_iV_j+V_iH_j+
V_i^AV_j^B+V_i^BV_j^A
\right].
}
\]

The first two terms move infectivity between plants. The last two exchange acquisition-origin labels.

The full state-changing Gillespie intensity can therefore be written as

\[
\begin{aligned}
\Lambda_{\mathrm{tot}}(t)
=&
\sum_i
\Big[
\beta_{v_i}V_i\mathbf 1_{\{X_i=S\}}
+\gamma_{v_i}\mathbf 1_{\{X_i=L\}}
+\rho_{v_i}\mathbf 1_{\{X_i=I\}}\\
&\qquad
+\alpha_{v_i}H_i\mathbf 1_{\{X_i=I\}}
+\omega V_i
+rV_i
\Big]
+\underbrace{\Lambda_{\mathrm{disp}}(t)}_{\textbf{dispersion}}.
\end{aligned}
\]

The implementation below uses an exact rejection/thinning representation of this generator that avoids enumerating all pair propensities after every epidemiological event.


In [ ]:
def lattice_coordinates(n_rows: int, n_cols: int, spacing: float = 1.0) -> np.ndarray:
    """Coordinates in plant-spacing units (or metres if spacing is in metres)."""
    return np.array(
        [(r * spacing, c * spacing) for r in range(n_rows) for c in range(n_cols)],
        dtype=float,
    )


def pairwise_distances(coords: np.ndarray) -> np.ndarray:
    delta = coords[:, None, :] - coords[None, :, :]
    return np.sqrt(np.sum(delta * delta, axis=2))


def symmetric_sinkhorn(
    weights: np.ndarray,
    tol: float = 1e-12,
    max_iter: int = 20_000,
) -> np.ndarray:
    """Balance a symmetric non-negative matrix to a symmetric doubly stochastic matrix.

    Standard alternating Sinkhorn scaling is used. For a symmetric matrix with
    total support, the balanced matrix is symmetric up to floating-point error.
    """
    W = np.asarray(weights, dtype=float)
    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError("weights must be square.")
    if np.any(W < 0):
        raise ValueError("weights must be non-negative.")
    if np.any(W.sum(axis=1) <= 0):
        raise ValueError("Every row must have positive support.")

    n = W.shape[0]
    r = np.ones(n)
    c = np.ones(n)

    for iteration in range(max_iter):
        r = 1.0 / (W @ c)
        c = 1.0 / (W.T @ r)

        if iteration % 10 == 0:
            P = (r[:, None] * W) * c[None, :]
            row_error = np.max(np.abs(P.sum(axis=1) - 1.0))
            col_error = np.max(np.abs(P.sum(axis=0) - 1.0))
            if max(row_error, col_error) < tol:
                break
    else:
        raise RuntimeError("Sinkhorn balancing did not converge.")

    P = (r[:, None] * W) * c[None, :]

    # The theoretical result is symmetric. Average only at floating-point scale,
    # then perform one final balancing pass if needed.
    symmetry_error = np.max(np.abs(P - P.T))
    if symmetry_error > 1e-9:
        raise RuntimeError(
            f"Balanced kernel is unexpectedly asymmetric: {symmetry_error:.3e}"
        )

    P = 0.5 * (P + P.T)

    # Tiny averaging perturbations are removed by a second symmetric balancing.
    if np.max(np.abs(P.sum(axis=1) - 1.0)) > tol:
        return symmetric_sinkhorn(P, tol=tol, max_iter=max_iter)

    np.fill_diagonal(P, 0.0)
    return P


def prepare_kernel(
    n_rows: int,
    n_cols: int,
    kernel_scale: float,
    spacing: float = 1.0,
):
    """Build the balanced exponential movement kernel and diagnostics."""
    if kernel_scale <= 0:
        raise ValueError("kernel_scale must be positive.")

    coords = lattice_coordinates(n_rows, n_cols, spacing=spacing)
    distances = pairwise_distances(coords)

    W = np.exp(-distances / kernel_scale)
    np.fill_diagonal(W, 0.0)

    P = symmetric_sinkhorn(W)

    row_sums = P.sum(axis=1)
    mean_distance = float(np.mean(np.sum(P * distances, axis=1)))

    cdf = np.cumsum(P, axis=1)
    cdf[:, -1] = 1.0

    M = n_rows * n_cols
    pair_mass = float(np.sum(np.triu(P, k=1)))

    return {
        "coordinates": coords,
        "distances": distances,
        "probabilities": P,
        "cdf": cdf,
        "mean_step_distance": mean_distance,
        "pair_mass": pair_mass,
        "max_row_error": float(np.max(np.abs(row_sums - 1.0))),
        "max_symmetry_error": float(np.max(np.abs(P - P.T))),
        "field_diameter": float(np.max(distances)),
        "expected_pair_mass": M / 2.0,
    }


# Geometry diagnostic over the exact requested 100-point domain.
N = 10
SENSITIVITY_GRID = np.geomspace(0.05, 100.0, 100)

kernel_geometry_rows = []
for a in SENSITIVITY_GRID:
    k = prepare_kernel(N, N, float(a))
    kernel_geometry_rows.append({
        "a": float(a),
        "mean_step_distance": k["mean_step_distance"],
        "pair_mass": k["pair_mass"],
        "max_row_error": k["max_row_error"],
        "max_symmetry_error": k["max_symmetry_error"],
    })

kernel_geometry = pd.DataFrame(kernel_geometry_rows)

plt.figure(figsize=(8.5, 5.0))
plt.plot(kernel_geometry["a"], kernel_geometry["mean_step_distance"])
plt.axvline(np.sqrt(2) * (N - 1), linestyle=":", label="field diameter")
plt.xscale("log")
plt.xlabel("Exponential kernel scale a (plant spacings)")
plt.ylabel("Finite-field mean step distance")
plt.title("Kernel scale and implied mean movement distance")
plt.legend()
plt.tight_layout()
plt.show()

kernel_geometry.head()


## 6.5 Efficient exact-state thinning used in the simulator

Direct simulation of all physical pair events would waste most CPU time on exchanges that do not change the tracked epidemic state.

Because \(P_{ij}=P_{ji}\), an equivalent proposal representation is available.

For a viruliferous vector at \(i\):

1. propose movement at rate \(\sigma\);
2. choose \(j\) with probability \(P_{ij}\);
3. select the reciprocal vector uniformly at \(j\).

For a viruliferous–healthy pair, the resulting transition rate is

\[
\sigma V_iP_{ij}\frac{H_j}{m}
=
\frac{\sigma}{m}P_{ij}V_iH_j,
\]

which is exactly the rate induced by the physical pair generator.

When two viruliferous vectors with different acquisition origins are selected, the source-proposal representation counts the same unordered physical swap from both directions. Such provenance exchanges are therefore accepted with probability \(1/2\). Same-status exchanges are null.

This is an exact thinning/rejection representation for the **tracked state process**; it is not an approximation to the pairwise generator.

The simulator also omits healthy-vector mortality/replacement because that event leaves all tracked state variables unchanged.


In [ ]:
# ---------------------------------------------------------------------
# Compiled stochastic simulator
# ---------------------------------------------------------------------
S_STATE = np.int8(0)
L_STATE = np.int8(1)
I_STATE = np.int8(2)


@njit
def _weighted_index(weights, total, u):
    target = u * total
    cumulative = 0.0
    for idx in range(weights.shape[0]):
        cumulative += weights[idx]
        if cumulative >= target:
            return idx
    return weights.shape[0] - 1


@njit
def _sample_cdf_row(cdf, row, u):
    low = 0
    high = cdf.shape[1] - 1
    while low < high:
        mid = (low + high) // 2
        if u <= cdf[row, mid]:
            high = mid
        else:
            low = mid + 1
    return low


@njit
def _single_ssa_trajectory(
    cultivar,
    destination_cdf,
    vectors_per_plant,
    initial_sites,
    observation_times,
    t_end,
    event_seed,
    acquisition_a,
    acquisition_b,
    inoculation_a,
    inoculation_b,
    gamma_a,
    gamma_b,
    rho_a,
    rho_b,
    sigma,
    omega,
    clearance,
    yield_healthy_a,
    yield_infected_a,
    yield_healthy_b,
    yield_infected_b,
):
    """Exact rejection/thinning SSA for the tracked epidemic state."""
    np.random.seed(event_seed)

    n_cells = cultivar.shape[0]
    n_obs = observation_times.shape[0]

    plant_state = np.zeros(n_cells, dtype=np.int8)
    for z in range(initial_sites.shape[0]):
        plant_state[initial_sites[z]] = I_STATE

    # All vectors initially virus-free.
    vir_a = np.zeros(n_cells, dtype=np.int16)
    vir_b = np.zeros(n_cells, dtype=np.int16)

    # Running counts avoid rescanning plant states at every observation time.
    infectious_a = 0
    infectious_b = 0
    for z in range(initial_sites.shape[0]):
        site = initial_sites[z]
        if cultivar[site] == 0:
            infectious_a += 1
        else:
            infectious_b += 1

    vir_total_count = 0

    incidence = np.empty(n_obs, dtype=np.float64)
    incidence_a = np.empty(n_obs, dtype=np.float64)
    incidence_b = np.empty(n_obs, dtype=np.float64)
    vector_prevalence = np.empty(n_obs, dtype=np.float64)

    # Per-site temporary propensities.
    inoculation_rates = np.empty(n_cells, dtype=np.float64)
    progression_rates = np.empty(n_cells, dtype=np.float64)
    roguing_rates = np.empty(n_cells, dtype=np.float64)
    acquisition_rates = np.empty(n_cells, dtype=np.float64)
    vir_counts = np.empty(n_cells, dtype=np.float64)

    obs_idx = 0
    time_now = 0.0

    while time_now < t_end:
        inoculation_total = 0.0
        progression_total = 0.0
        roguing_total = 0.0
        acquisition_total = 0.0

        for i in range(n_cells):
            v_i = vir_a[i] + vir_b[i]
            vir_counts[i] = v_i

            if plant_state[i] == S_STATE:
                beta = inoculation_a if cultivar[i] == 0 else inoculation_b
                inoculation_rates[i] = beta * v_i
            else:
                inoculation_rates[i] = 0.0
            inoculation_total += inoculation_rates[i]

            if plant_state[i] == L_STATE:
                progression_rates[i] = gamma_a if cultivar[i] == 0 else gamma_b
            else:
                progression_rates[i] = 0.0
            progression_total += progression_rates[i]

            if plant_state[i] == I_STATE:
                roguing_rates[i] = rho_a if cultivar[i] == 0 else rho_b
                alpha = acquisition_a if cultivar[i] == 0 else acquisition_b
                healthy = vectors_per_plant - v_i
                acquisition_rates[i] = alpha * healthy
            else:
                roguing_rates[i] = 0.0
                acquisition_rates[i] = 0.0

            roguing_total += roguing_rates[i]
            acquisition_total += acquisition_rates[i]

        # Efficient exact-state movement proposal.
        dispersion_proposal_total = sigma * vir_total_count
        mortality_total = omega * vir_total_count
        clearance_total = clearance * vir_total_count

        total_rate = (
            inoculation_total
            + progression_total
            + roguing_total
            + acquisition_total
            + dispersion_proposal_total
            + mortality_total
            + clearance_total
        )

        if total_rate <= 0.0:
            # No future tracked-state transition is possible.
            while obs_idx < n_obs:
                incidence[obs_idx] = (infectious_a + infectious_b) / n_cells
                incidence_a[obs_idx] = infectious_a / n_cells
                incidence_b[obs_idx] = infectious_b / n_cells
                vector_prevalence[obs_idx] = (
                    vir_total_count / (vectors_per_plant * n_cells)
                )
                obs_idx += 1
            time_now = t_end
            break

        waiting = -math.log(max(np.random.random(), 1e-15)) / total_rate
        event_time = time_now + waiting

        # Record the left-continuous state at all observation times strictly
        # before the next event.
        while obs_idx < n_obs and observation_times[obs_idx] < event_time:
            incidence[obs_idx] = (infectious_a + infectious_b) / n_cells
            incidence_a[obs_idx] = infectious_a / n_cells
            incidence_b[obs_idx] = infectious_b / n_cells
            vector_prevalence[obs_idx] = (
                vir_total_count / (vectors_per_plant * n_cells)
            )
            obs_idx += 1

        if event_time > t_end:
            time_now = t_end
            break

        time_now = event_time
        draw = np.random.random() * total_rate
        threshold = inoculation_total

        # -------------------------------------------------------------
        # Plant inoculation: S -> L
        # -------------------------------------------------------------
        if draw < threshold:
            i = _weighted_index(
                inoculation_rates, inoculation_total, np.random.random()
            )
            plant_state[i] = L_STATE
            continue

        # -------------------------------------------------------------
        # End of plant latency: L -> I
        # -------------------------------------------------------------
        threshold += progression_total
        if draw < threshold:
            i = _weighted_index(
                progression_rates, progression_total, np.random.random()
            )
            plant_state[i] = I_STATE
            if cultivar[i] == 0:
                infectious_a += 1
            else:
                infectious_b += 1
            continue

        # -------------------------------------------------------------
        # Roguing: I -> S
        # -------------------------------------------------------------
        threshold += roguing_total
        if draw < threshold:
            i = _weighted_index(roguing_rates, roguing_total, np.random.random())
            plant_state[i] = S_STATE
            if cultivar[i] == 0:
                infectious_a -= 1
            else:
                infectious_b -= 1
            continue

        # -------------------------------------------------------------
        # Virus acquisition by a healthy vector on an infectious plant
        # -------------------------------------------------------------
        threshold += acquisition_total
        if draw < threshold:
            i = _weighted_index(
                acquisition_rates, acquisition_total, np.random.random()
            )
            if cultivar[i] == 0:
                vir_a[i] += 1
            else:
                vir_b[i] += 1
            vir_total_count += 1
            continue

        # -------------------------------------------------------------
        # Dispersal proposal, exactly thinned to the pair-swap state process
        # -------------------------------------------------------------
        threshold += dispersion_proposal_total
        if draw < threshold:
            if vir_total_count <= 0:
                continue

            source = _weighted_index(
                vir_counts, float(vir_total_count), np.random.random()
            )

            source_total = vir_a[source] + vir_b[source]
            if source_total <= 0:
                continue

            moving_origin = (
                0
                if np.random.random() < vir_a[source] / source_total
                else 1
            )

            destination = _sample_cdf_row(
                destination_cdf, source, np.random.random()
            )

            h_dest = (
                vectors_per_plant
                - vir_a[destination]
                - vir_b[destination]
            )
            reciprocal_draw = np.random.random() * vectors_per_plant

            if reciprocal_draw < h_dest:
                # Viruliferous <-> healthy. This changes the location of infectivity.
                if moving_origin == 0:
                    vir_a[source] -= 1
                    vir_a[destination] += 1
                else:
                    vir_b[source] -= 1
                    vir_b[destination] += 1

            elif reciprocal_draw < h_dest + vir_a[destination]:
                # Reciprocal vector is origin-A viruliferous.
                # A<->A is state-null.
                # B<->A is counted by source proposals from both directions;
                # accept with probability 1/2 to recover the unordered pair rate.
                if moving_origin == 1 and np.random.random() < 0.5:
                    vir_b[source] -= 1
                    vir_a[source] += 1
                    vir_a[destination] -= 1
                    vir_b[destination] += 1

            else:
                # Reciprocal vector is origin-B viruliferous.
                if moving_origin == 0 and np.random.random() < 0.5:
                    vir_a[source] -= 1
                    vir_b[source] += 1
                    vir_b[destination] -= 1
                    vir_a[destination] += 1

            continue

        # -------------------------------------------------------------
        # Viruliferous-vector mortality + healthy replacement
        # -------------------------------------------------------------
        threshold += mortality_total
        if draw < threshold:
            i = _weighted_index(
                vir_counts, float(vir_total_count), np.random.random()
            )
            total_here = vir_a[i] + vir_b[i]
            if np.random.random() < vir_a[i] / total_here:
                vir_a[i] -= 1
            else:
                vir_b[i] -= 1
            vir_total_count -= 1
            continue

        # -------------------------------------------------------------
        # Loss of vector infectivity
        # -------------------------------------------------------------
        i = _weighted_index(
            vir_counts, float(vir_total_count), np.random.random()
        )
        total_here = vir_a[i] + vir_b[i]
        if np.random.random() < vir_a[i] / total_here:
            vir_a[i] -= 1
        else:
            vir_b[i] -= 1
        vir_total_count -= 1

    # Fill observations at or after the final event.
    while obs_idx < n_obs:
        incidence[obs_idx] = (infectious_a + infectious_b) / n_cells
        incidence_a[obs_idx] = infectious_a / n_cells
        incidence_b[obs_idx] = infectious_b / n_cells
        vector_prevalence[obs_idx] = vir_total_count / (
            vectors_per_plant * n_cells
        )
        obs_idx += 1

    # Final area-normalised yield.
    final_yield = 0.0
    for i in range(n_cells):
        infected = plant_state[i] == I_STATE
        if cultivar[i] == 0:
            final_yield += (
                yield_infected_a if infected else yield_healthy_a
            )
        else:
            final_yield += (
                yield_infected_b if infected else yield_healthy_b
            )
    final_yield /= n_cells

    return (
        final_yield,
        (infectious_a + infectious_b) / n_cells,
        plant_state,
        vir_a,
        vir_b,
        incidence,
        incidence_a,
        incidence_b,
        vector_prevalence,
    )


@njit(parallel=True)
def _batch_ssa_trajectory(
    cultivar,
    destination_cdf,
    vectors_per_plant,
    initial_sites_table,
    observation_times,
    t_end,
    event_seeds,
    acquisition_a,
    acquisition_b,
    inoculation_a,
    inoculation_b,
    gamma_a,
    gamma_b,
    rho_a,
    rho_b,
    sigma,
    omega,
    clearance,
    yield_healthy_a,
    yield_infected_a,
    yield_healthy_b,
    yield_infected_b,
):
    n_runs = event_seeds.shape[0]
    n_cells = cultivar.shape[0]
    n_obs = observation_times.shape[0]

    yields = np.empty(n_runs, dtype=np.float64)
    final_incidence = np.empty(n_runs, dtype=np.float64)
    final_states = np.empty((n_runs, n_cells), dtype=np.int8)
    trajectories_i = np.empty((n_runs, n_obs), dtype=np.float64)
    trajectories_ia = np.empty((n_runs, n_obs), dtype=np.float64)
    trajectories_ib = np.empty((n_runs, n_obs), dtype=np.float64)
    trajectories_v = np.empty((n_runs, n_obs), dtype=np.float64)

    for run in prange(n_runs):
        result = _single_ssa_trajectory(
            cultivar,
            destination_cdf,
            vectors_per_plant,
            initial_sites_table[run],
            observation_times,
            t_end,
            int(event_seeds[run]),
            acquisition_a,
            acquisition_b,
            inoculation_a,
            inoculation_b,
            gamma_a,
            gamma_b,
            rho_a,
            rho_b,
            sigma,
            omega,
            clearance,
            yield_healthy_a,
            yield_infected_a,
            yield_healthy_b,
            yield_infected_b,
        )

        yields[run] = result[0]
        final_incidence[run] = result[1]
        final_states[run] = result[2]
        trajectories_i[run] = result[5]
        trajectories_ia[run] = result[6]
        trajectories_ib[run] = result[7]
        trajectories_v[run] = result[8]

    return (
        yields,
        final_incidence,
        final_states,
        trajectories_i,
        trajectories_ia,
        trajectories_ib,
        trajectories_v,
    )


def make_initial_sites(
    n_cells: int,
    n_initial_infectious: int,
    n_runs: int,
    seed: int,
) -> np.ndarray:
    """Generate reproducible, without-replacement initial infection locations."""
    if not 1 <= n_initial_infectious <= n_cells:
        raise ValueError("Invalid number of initially infectious plants.")

    rng = np.random.default_rng(seed)
    sites = np.empty((n_runs, n_initial_infectious), dtype=np.int64)
    for run in range(n_runs):
        sites[run] = rng.choice(
            n_cells, size=n_initial_infectious, replace=False
        )
    return sites


def run_spatial_ensemble(
    cultivar_grid: np.ndarray,
    kernel: dict,
    vectors_per_plant: int,
    n_initial_infectious: int,
    n_runs: int,
    initial_site_seed: int,
    event_seed: int,
    observation_times: np.ndarray = OBS_TIMES,
    t_end: float = T_END,
    transmission_draw: MixtureTransmissionDraw | None = None,
):
    """Validated wrapper around the compiled stochastic spatial engine."""
    if transmission_draw is None:
        transmission_draw = POINT_TRANSMISSION_DRAW

    if cultivar_grid.ndim != 2:
        raise ValueError("cultivar_grid must be two-dimensional.")
    if vectors_per_plant <= 0:
        raise ValueError("vectors_per_plant must be positive.")

    n_rows, n_cols = cultivar_grid.shape
    n_cells = n_rows * n_cols
    cultivar = np.asarray(cultivar_grid, dtype=np.int8).ravel()

    initial_sites = make_initial_sites(
        n_cells,
        n_initial_infectious,
        n_runs,
        initial_site_seed,
    )
    event_seeds = np.arange(
        event_seed, event_seed + n_runs, dtype=np.int64
    )

    result = _batch_ssa_trajectory(
        cultivar,
        kernel["cdf"],
        vectors_per_plant,
        initial_sites,
        np.asarray(observation_times, dtype=float),
        t_end,
        event_seeds,
        transmission_draw.susc_acquisition,
        transmission_draw.res_acquisition,
        transmission_draw.susc_inoculation,
        transmission_draw.res_inoculation,
        SUSC.latency_progression,
        RES.latency_progression,
        SUSC.roguing,
        RES.roguing,
        SIGMA,
        OMEGA,
        transmission_draw.vector_clearance,
        SUSC.yield_healthy,
        SUSC.yield_infected,
        RES.yield_healthy,
        RES.yield_infected,
    )

    (
        yields,
        final_incidence,
        final_states,
        incidence,
        incidence_a,
        incidence_b,
        vector_prevalence,
    ) = result

    return {
        "yields": yields,
        "final_incidence": final_incidence,
        "final_states": final_states,
        "incidence": incidence,
        "incidence_a": incidence_a,
        "incidence_b": incidence_b,
        "vector_prevalence": vector_prevalence,
        "infection_probability": np.mean(
            final_states == I_STATE, axis=0
        ).reshape(n_rows, n_cols),
        "initial_sites": initial_sites,
        "kernel_scale": None,
        "mean_step_distance": kernel["mean_step_distance"],
    }

## 6.6 Optional posterior-predictive propagation from EpiPvr into the completed spatial CTMC

The spatial engine accepts one coherent transmission draw. If joint EpiPvr posterior samples are available, nested Monte Carlo samples \(\Theta^{(b)}\) from the posterior and then multiple Gillespie trajectories conditional on each draw.

For yield,

\[
E[Y\mid D,z]
\approx
\frac1B\sum_b\frac1R\sum_rY^{(b,r)}.
\]

The helper below is defined but not executed unless posterior samples are supplied.


In [ ]:
def posterior_predictive_spatial(
    cultivar_grid,
    kernel,
    posterior_samples,
    vectors_per_plant,
    n_initial_infectious,
    n_parameter_draws=100,
    n_process_runs_per_draw=20,
    seed=MASTER_SEED + 9_000_000,
    observation_times=OBS_TIMES,
):
    """Nested posterior/process Monte Carlo for the current two-variety SPT model."""
    if posterior_samples is None or len(posterior_samples) == 0:
        raise ValueError("posterior_samples must contain at least one draw.")

    rng = np.random.default_rng(seed)
    rows = rng.integers(0, len(posterior_samples), size=n_parameter_draws)
    records = []

    for b, row_index in enumerate(rows):
        draw = posterior_draw_from_row(posterior_samples.iloc[int(row_index)])
        result = run_spatial_ensemble(
            cultivar_grid,
            kernel,
            vectors_per_plant=vectors_per_plant,
            n_initial_infectious=n_initial_infectious,
            n_runs=n_process_runs_per_draw,
            initial_site_seed=seed + 100_000 + b * 10_000,
            event_seed=seed + 200_000 + b * 10_000,
            observation_times=observation_times,
            transmission_draw=draw,
        )
        records.append({
            "posterior_row": int(row_index),
            "mean_yield_given_draw": float(result["yields"].mean()),
            "mean_final_incidence_given_draw": float(result["final_incidence"].mean()),
            "susc_acquisition": draw.susc_acquisition,
            "susc_inoculation": draw.susc_inoculation,
            "res_acquisition": draw.res_acquisition,
            "res_inoculation": draw.res_inoculation,
            "vector_clearance": draw.vector_clearance,
        })

    return pd.DataFrame(records)



# 7. Mathematical validation of the movement generator

Before any epidemiological calibration, the spatial movement model must satisfy properties that are independent of disease outcomes.

For each tested \(a\):

1. \(P_{ii}=0\);
2. \(P_{ij}=P_{ji}\);
3. \(\sum_jP_{ij}=1\);
4. \(\sum_{i<j}P_{ij}=M/2\);
5. a tagged vector's total physical movement hazard is \(\sigma\);
6. the physical total pair-event rate is \(Mm\sigma/2\).

These checks are more fundamental than matching an epidemic curve.


In [ ]:
# Mathematical kernel audit at representative scales.
movement_audit_rows = []
for a in (0.05, 0.5, 1.0, 5.0, 20.0, 100.0):
    k = prepare_kernel(N, N, a)
    P = k["probabilities"]
    tagged_rates = SIGMA * P.sum(axis=1)

    movement_audit_rows.append({
        "a": a,
        "max_abs_diagonal": float(np.max(np.abs(np.diag(P)))),
        "max_row_error": float(np.max(np.abs(P.sum(axis=1) - 1.0))),
        "max_symmetry_error": float(np.max(np.abs(P - P.T))),
        "pair_mass": float(np.triu(P, 1).sum()),
        "expected_pair_mass": (N * N) / 2.0,
        "min_tagged_rate": float(np.min(tagged_rates)),
        "max_tagged_rate": float(np.max(tagged_rates)),
        "target_sigma": SIGMA,
    })

movement_audit = pd.DataFrame(movement_audit_rows)
movement_audit

In [ ]:
# Compile the stochastic engine and run a very small invariant smoke test.
_smoke_grid = np.zeros((2, 2), dtype=np.int8)
_smoke_kernel = prepare_kernel(2, 2, 1.0)
_smoke_initial = np.array([0], dtype=np.int64)

_smoke = _single_ssa_trajectory(
    _smoke_grid.ravel(),
    _smoke_kernel["cdf"],
    3,
    _smoke_initial,
    np.linspace(0.0, 5.0, 6),
    5.0,
    MASTER_SEED,
    SUSC.acquisition,
    RES.acquisition,
    SUSC.inoculation,
    RES.inoculation,
    SUSC.latency_progression,
    RES.latency_progression,
    SUSC.roguing,
    RES.roguing,
    SIGMA,
    OMEGA,
    VECTOR_CLEARANCE,
    SUSC.yield_healthy,
    SUSC.yield_infected,
    RES.yield_healthy,
    RES.yield_infected,
)

_smoke_vir_a = _smoke[3]
_smoke_vir_b = _smoke[4]
assert np.all(_smoke_vir_a >= 0)
assert np.all(_smoke_vir_b >= 0)
assert np.all(_smoke_vir_a + _smoke_vir_b <= 3)
assert np.isin(_smoke[2], [S_STATE, L_STATE, I_STATE]).all()
print("Compiled SSA smoke test passed.")

# 8. Mean-field approximation, branching-process consistency, and robust kernel-scale selection

Two consistency exercises must be separated.

## 8.1 Branching-process consistency is an early-invasion validation problem

A future validation experiment can compare the EpiPvr branching-process epidemic probability with Monte Carlo establishment probability from a harmonised Gillespie model.

The current arrangement experiment has no roguing and only terminal harvest. An initially infectious plant persists during the season. That is not the same extinction structure as the EpiPvr branching process.

A valid numerical comparison therefore requires a separate harmonised scenario with matching vector burden, transmission rates, mortality, clearance, dispersal, plant latent progression, roguing/removal, harvest, SPT/PT compartment structure, host homogeneity and establishment definition.

The notebook does **not** relabel an arbitrary finite-season threshold probability as the EpiPvr epidemic probability.

## 8.2 The calibration performed here is PLOS mean-field consistency

The present study instead asks:

> What explicit spatial mixing scale gives the best joint consistency with the deterministic PLOS model across several epidemic-pressure contexts?

Thus

\[
\boxed{
\text{branching-process comparison}
\neq
\text{mean-field kernel calibration}.
}
\]



## 8.3 What can and cannot be inferred

The exponential scale \(a\) describes vector movement. A biological movement parameter should not change merely because the initial number of vectors or infectious plants changes.

Therefore, values

\[
a_s^*=
\arg\min_a D_s(a)
\]

calculated separately for epidemic scenarios \(s\) are treated as **diagnostics**. They are not separate estimates of a biological dispersal scale.

The primary question is instead:

> Is there a **single** \(a\) for which the spatial model is jointly compatible with the corresponding mean-field dynamics over several epidemiological contexts?

---

## 8.4 Why final yield is no longer the calibration objective

In a susceptible monoculture,

\[
Y(T)=Y_H\{1-I(T)\}+Y_I I(T),
\]

so terminal yield contains almost the same information as final incidence. Optimising a score that includes both final incidence and yield would partially double-count one endpoint.

The primary consistency score therefore uses:

1. the entire plant-incidence trajectory;
2. the entire viruliferous-vector prevalence trajectory.

Final yield is reported as a **validation quantity**.

For scenario \(s\), define

\[
D_{I,s}(a)
=
\sqrt{
\frac{1}{J}
\sum_{j=1}^{J}
\left[
\bar I_{\mathrm{sp},s}(t_j;a)
-I_{\mathrm{MF},s}(t_j)
\right]^2
},
\]

and

\[
D_{V,s}(a)
=
\sqrt{
\frac{1}{J}
\sum_{j=1}^{J}
\left[
\bar V_{\mathrm{sp},s}(t_j;a)
-V_{\mathrm{MF},s}(t_j)
\right]^2
}.
\]

Both quantities are dimensionless and bounded on the same probability scale. The default scenario score is

\[
\boxed{
D_s(a)=\frac12D_{I,s}(a)+\frac12D_{V,s}(a).
}
\]

The joint all-context score is

\[
\boxed{
D_{\mathrm{joint}}(a)
=
\frac{1}{S}\sum_{s=1}^{S}D_s(a).
}
\]

A minimax diagnostic is also retained:

\[
D_{\max}(a)=\max_s D_s(a).
\]

---

## 8.5 Calibration contexts

The three core contexts reproduce the PLOS vector-pressure levels while retaining one introduced infectious plant:

- \(m=1,\ I_0=1\);
- \(m=5,\ I_0=1\);
- \(m=10,\ I_0=1\).

Two additional stress tests vary disease pressure while holding vector pressure low:

- \(m=1,\ I_0=5\);
- \(m=1,\ I_0=10\).

These larger inocula are **prototype robustness scenarios**, not parameter values taken from the PLOS article.

For the \(10\times10\) prototype, the matched deterministic reference uses \(K=100\). The literal \(K=10{,}000\) PLOS result remains an external benchmark and must be revisited at publication-scale \(100\times100\).

---

## 8.6 Selection rule

Three quantities are reported:

**Literal joint optimum**

\[
a_{\min}=\arg\min_aD_{\mathrm{joint}}(a).
\]

**One-standard-error operational scale**

\[
a_{\mathrm{MF}}
=
\min\left\{
a:
D_{\mathrm{joint}}(a)
\le
D_{\mathrm{joint}}(a_{\min})
+
\mathrm{SE}\left[D_{\mathrm{joint}}(a_{\min})\right]
\right\}.
\]

This selects the smallest spatial scale already statistically compatible with the best mean-field agreement, rather than an arbitrary point deep in a flat well-mixed plateau.

**Acceptable scale region**

All scanned \(a\) values satisfying the same one-SE criterion are retained. Its width is an identifiability diagnostic.

A broad acceptable region means that \(a\) is weakly identifiable from mean-field consistency, even if the spatial model is compatible with the mean-field model.


In [ ]:
# ---------------------------------------------------------------------
# Calibration configuration
# ---------------------------------------------------------------------
SENSITIVITY_GRID = np.geomspace(0.05, 100.0, 100)

CALIBRATION_CONTEXTS = pd.DataFrame([
    ["low vector / introduction",      1,  1, "PLOS-core"],
    ["medium vector / introduction",   5,  1, "PLOS-core"],
    ["high vector / introduction",    10,  1, "PLOS-core"],
    ["low vector / moderate inoculum", 1,  5, "stress-test"],
    ["low vector / high inoculum",     1, 10, "stress-test"],
], columns=["context", "vectors_per_plant", "initial_infectious", "role"])

CALIBRATION_CONTEXTS

In [ ]:
# Precompute all 100 movement kernels once and reuse them in every context.
kernel_cache = {}
kernel_start = time.time()

for a in SENSITIVITY_GRID:
    kernel_cache[float(a)] = prepare_kernel(N, N, float(a))

print(f"Prepared {len(kernel_cache)} kernels in {time.time() - kernel_start:.2f} s.")

In [ ]:
def trajectory_score(
    spatial_incidence_mean: np.ndarray,
    spatial_vector_mean: np.ndarray,
    meanfield_incidence: np.ndarray,
    meanfield_vector: np.ndarray,
):
    """Dimensionless dynamic consistency score."""
    rmse_i = float(np.sqrt(np.mean(
        (spatial_incidence_mean - meanfield_incidence) ** 2
    )))
    rmse_v = float(np.sqrt(np.mean(
        (spatial_vector_mean - meanfield_vector) ** 2
    )))
    score = 0.5 * rmse_i + 0.5 * rmse_v
    return score, rmse_i, rmse_v


def bootstrap_trajectory_score_se(
    incidence_runs: np.ndarray,
    vector_runs: np.ndarray,
    meanfield_incidence: np.ndarray,
    meanfield_vector: np.ndarray,
    n_bootstrap: int,
    seed: int,
):
    """Bootstrap Monte Carlo uncertainty in the score of the ensemble mean."""
    n_runs = incidence_runs.shape[0]
    if n_runs < 2:
        return np.nan

    rng = np.random.default_rng(seed)
    scores = np.empty(n_bootstrap, dtype=float)

    for b in range(n_bootstrap):
        sample = rng.integers(0, n_runs, size=n_runs)
        mean_i = incidence_runs[sample].mean(axis=0)
        mean_v = vector_runs[sample].mean(axis=0)
        scores[b] = trajectory_score(
            mean_i, mean_v, meanfield_incidence, meanfield_vector
        )[0]

    return float(np.std(scores, ddof=1))


def run_kernel_consistency_scan():
    """Run the full 100-point, multi-context mean-field consistency experiment."""
    susceptible_monoculture = np.zeros((N, N), dtype=np.int8)
    records = []

    start = time.time()

    for context_idx, context in CALIBRATION_CONTEXTS.iterrows():
        m = int(context["vectors_per_plant"])
        i0 = int(context["initial_infectious"])

        mf = solve_plos_meanfield(
            theta=1.0,
            vector_burden=m,
            K=N * N,
            n_initial_infectious=i0,
            t_eval=OBS_TIMES,
        )

        # Common initial infection locations across all a in this context.
        initial_seed = MASTER_SEED + 10_000 + 1_000 * context_idx
        # Common event seed block across all a in this context.
        event_seed = MASTER_SEED + 100_000 + 10_000 * context_idx

        for a_idx, a in enumerate(SENSITIVITY_GRID):
            a = float(a)
            result = run_spatial_ensemble(
                susceptible_monoculture,
                kernel_cache[a],
                vectors_per_plant=m,
                n_initial_infectious=i0,
                n_runs=N_CALIBRATION_RUNS,
                initial_site_seed=initial_seed,
                event_seed=event_seed,
                observation_times=OBS_TIMES,
            )

            spatial_i = result["incidence"].mean(axis=0)
            spatial_v = result["vector_prevalence"].mean(axis=0)

            score, rmse_i, rmse_v = trajectory_score(
                spatial_i,
                spatial_v,
                mf["incidence"],
                mf["vector_prevalence"],
            )

            score_se = bootstrap_trajectory_score_se(
                result["incidence"],
                result["vector_prevalence"],
                mf["incidence"],
                mf["vector_prevalence"],
                n_bootstrap=N_BOOTSTRAP,
                seed=MASTER_SEED + 1_000_000 + context_idx * 10_000 + a_idx,
            )

            spatial_yield = float(result["yields"].mean())
            yield_error = abs(spatial_yield - mf["yield"])
            relative_yield_error = yield_error / max(abs(mf["yield"]), 1e-12)

            records.append({
                "context": context["context"],
                "role": context["role"],
                "vectors_per_plant": m,
                "initial_infectious": i0,
                "a": a,
                "mean_step_distance": kernel_cache[a]["mean_step_distance"],
                "dynamic_score": score,
                "score_se": score_se,
                "incidence_rmse": rmse_i,
                "vector_prevalence_rmse": rmse_v,
                "spatial_final_yield": spatial_yield,
                "meanfield_final_yield": mf["yield"],
                "absolute_yield_error": yield_error,
                "relative_yield_error": relative_yield_error,
                "spatial_final_incidence": float(result["final_incidence"].mean()),
                "meanfield_final_incidence": mf["final_incidence"],
            })

        print(
            f"Completed {context['context']} "
            f"({context_idx + 1}/{len(CALIBRATION_CONTEXTS)})"
        )

    print(f"Calibration scan runtime: {time.time() - start:.1f} s")
    return pd.DataFrame(records)


# This is intentionally a substantial calculation: 100 kernel scales are
# evaluated in every calibration context.
calibration_long = run_kernel_consistency_scan()
calibration_long.head()

In [ ]:
def aggregate_joint_scores(calibration_long: pd.DataFrame, role_filter=None):
    data = calibration_long
    if role_filter is not None:
        data = data[data["role"].isin(role_filter)].copy()

    rows = []
    for a, group in data.groupby("a", sort=True):
        ses = group["score_se"].to_numpy(float)
        finite_ses = ses[np.isfinite(ses)]

        joint_se = (
            float(np.sqrt(np.sum(finite_ses ** 2)) / len(group))
            if finite_ses.size
            else np.nan
        )

        rows.append({
            "a": float(a),
            "mean_step_distance": float(group["mean_step_distance"].iloc[0]),
            "joint_mean_score": float(group["dynamic_score"].mean()),
            "joint_max_score": float(group["dynamic_score"].max()),
            "joint_score_se": joint_se,
            "mean_relative_yield_error": float(group["relative_yield_error"].mean()),
            "max_relative_yield_error": float(group["relative_yield_error"].max()),
            "n_contexts": len(group),
        })
    return pd.DataFrame(rows).sort_values("a").reset_index(drop=True)


joint_all = aggregate_joint_scores(calibration_long)
joint_core = aggregate_joint_scores(calibration_long, role_filter=["PLOS-core"])


def select_one_se_scale(joint_table: pd.DataFrame):
    best_idx = joint_table["joint_mean_score"].idxmin()
    best = joint_table.loc[best_idx]

    best_se = float(best["joint_score_se"])
    if not np.isfinite(best_se):
        best_se = 0.0

    threshold = float(best["joint_mean_score"]) + best_se
    acceptable = joint_table[joint_table["joint_mean_score"] <= threshold]

    selected = acceptable.iloc[0]

    return {
        "literal_a": float(best["a"]),
        "literal_score": float(best["joint_mean_score"]),
        "literal_score_se": best_se,
        "threshold": threshold,
        "selected_a": float(selected["a"]),
        "selected_mean_distance": float(selected["mean_step_distance"]),
        "acceptable_a_min": float(acceptable["a"].min()),
        "acceptable_a_max": float(acceptable["a"].max()),
        "acceptable_log10_width": float(
            np.log10(acceptable["a"].max() / acceptable["a"].min())
        ) if len(acceptable) > 1 else 0.0,
        "boundary_warning": bool(
            best["a"] == joint_table["a"].min()
            or best["a"] == joint_table["a"].max()
        ),
    }


selection_all = select_one_se_scale(joint_all)
selection_core = select_one_se_scale(joint_core)

SELECTED_KERNEL_SCALE = selection_all["selected_a"]
SELECTED_MEAN_STEP_DISTANCE = selection_all["selected_mean_distance"]

selection_summary = pd.DataFrame([
    {"scope": "all contexts", **selection_all},
    {"scope": "PLOS-core contexts only", **selection_core},
])

selection_summary

In [ ]:
# Context-specific optima are diagnostics only.
context_optima_rows = []
for context, group in calibration_long.groupby("context", sort=False):
    best = group.loc[group["dynamic_score"].idxmin()]
    context_optima_rows.append({
        "context": context,
        "role": best["role"],
        "vectors_per_plant": int(best["vectors_per_plant"]),
        "initial_infectious": int(best["initial_infectious"]),
        "literal_best_a": float(best["a"]),
        "best_dynamic_score": float(best["dynamic_score"]),
        "yield_error_at_best_a": float(best["absolute_yield_error"]),
    })

context_optima = pd.DataFrame(context_optima_rows)

# Leave-one-context-out robustness of the common optimum.
loco_rows = []
contexts = calibration_long["context"].drop_duplicates().tolist()
for omitted in contexts:
    subset = calibration_long[calibration_long["context"] != omitted]
    joint = aggregate_joint_scores(subset)
    sel = select_one_se_scale(joint)
    loco_rows.append({
        "omitted_context": omitted,
        "selected_a": sel["selected_a"],
        "literal_a": sel["literal_a"],
        "acceptable_a_min": sel["acceptable_a_min"],
        "acceptable_a_max": sel["acceptable_a_max"],
    })

loco_table = pd.DataFrame(loco_rows)

display(context_optima)
display(loco_table)


## 8.7 Interpreting robustness and identifiability

Three different phenomena must not be conflated:

### A. Context sensitivity

If the context-specific minima \(a_s^*\) differ, final epidemic outcomes do not identify \(a\) independently of epidemic pressure. This is expected when stochastic fade-out or epidemic saturation changes how strongly movement affects the endpoint.

### B. Weak identifiability

If the one-SE acceptable interval spans a large range of \(a\), the mean-field-consistency surface is flat. The correct conclusion is:

> the spatial model becomes sufficiently well mixed over a broad range of scales.

It is **not**:

> the biological whitefly dispersal scale has been measured precisely.

### C. Structural incompatibility

If even the best \(a\) leaves large incidence and vector-prevalence discrepancies, changing the distance kernel cannot make the spatial CTMC equivalent to the PLOS ODE. That is evidence for a structural difference between the microscopic spatialisation and the reduced mean-field construction.

This is a useful scientific result. It should trigger model revision or direct movement-data calibration rather than arbitrary tuning of \(a\).

Ultimately, a biological estimate of \(a\) should come from vector movement data or joint likelihood-based fitting to spatial epidemic observations. Mean-field matching is best interpreted as a **consistency constraint and operational scale-selection procedure**.


In [ ]:
# Main calibration plots
plt.figure(figsize=(9, 5.5))
plt.plot(joint_all["a"], joint_all["joint_mean_score"], label="Mean score: all contexts")
plt.plot(joint_all["a"], joint_all["joint_max_score"], label="Worst-context score")
plt.plot(joint_core["a"], joint_core["joint_mean_score"], linestyle="--", label="Mean score: PLOS-core only")
plt.axvline(SELECTED_KERNEL_SCALE, linestyle=":", label=f"selected a={SELECTED_KERNEL_SCALE:.4g}")
plt.xscale("log")
plt.xlabel("Exponential kernel scale a (plant spacings)")
plt.ylabel("Dynamic mean-field discrepancy")
plt.title("Cross-context mean-field consistency")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5.5))
for context, group in calibration_long.groupby("context", sort=False):
    plt.plot(group["a"], group["dynamic_score"], label=context)
plt.axvline(SELECTED_KERNEL_SCALE, linestyle=":")
plt.xscale("log")
plt.xlabel("Exponential kernel scale a (plant spacings)")
plt.ylabel("Scenario dynamic discrepancy")
plt.title("Context-specific identifiability of a")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5.5))
for context, group in calibration_long.groupby("context", sort=False):
    plt.plot(group["a"], group["relative_yield_error"], label=context)
plt.axvline(SELECTED_KERNEL_SCALE, linestyle=":")
plt.xscale("log")
plt.xlabel("Exponential kernel scale a (plant spacings)")
plt.ylabel("Relative final-yield error")
plt.title("Yield retained as an external validation outcome")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Compact summary table devoted to a.
a_summary_table = pd.DataFrame([
    {
        "quantity": "all-context literal minimum",
        "a": selection_all["literal_a"],
        "meaning": "Minimum cross-context dynamic discrepancy",
    },
    {
        "quantity": "all-context one-SE operational value",
        "a": selection_all["selected_a"],
        "meaning": "Single scale used for the primary arrangement analysis",
    },
    {
        "quantity": "acceptable lower bound",
        "a": selection_all["acceptable_a_min"],
        "meaning": "Lower edge of the one-SE consistency region",
    },
    {
        "quantity": "acceptable upper bound",
        "a": selection_all["acceptable_a_max"],
        "meaning": "Upper edge of the one-SE consistency region",
    },
    {
        "quantity": "PLOS-core operational value",
        "a": selection_core["selected_a"],
        "meaning": "Sensitivity to excluding the disease-pressure stress tests",
    },
])

a_summary_table


# 9. Planting arrangements

The manuscript comparison remains restricted to exact 50:50 `SUSC`–`RES` mixtures on an even square lattice.

Canonical patterns:

1. random exact 50:50 allocation;
2. checkerboard;
3. single-column vertical stripes;
4. quadrant split;
5. half split;
6. block checkerboard with \(k=2\);
7. block checkerboard with \(k=5\).

For \(N=10\), the \(k=5\) block checkerboard is exactly the quadrant design. It is displayed for didactic completeness but excluded as an independent data point from inferential metric analyses.

The same `SELECTED_KERNEL_SCALE` is used for every arrangement. No pattern is allowed to fit its own dispersal scale.


In [ ]:
def make_pattern(
    name: str,
    N: int,
    seed: int = MASTER_SEED,
    block_size: int = 2,
) -> np.ndarray:
    """Create an exact 50:50 SUSC(0)/RES(1) pattern."""
    if N % 2:
        raise ValueError("N must be even for an exact 50:50 mixture.")

    if name == "random":
        rng = np.random.default_rng(seed)
        values = np.array(
            [0] * (N * N // 2) + [1] * (N * N // 2),
            dtype=np.int8,
        )
        rng.shuffle(values)
        return values.reshape(N, N)

    if name == "checkerboard":
        rows, cols = np.indices((N, N))
        return ((rows + cols) % 2).astype(np.int8)

    if name == "vertical_stripes":
        return np.tile(np.arange(N) % 2, (N, 1)).astype(np.int8)

    if name == "quadrant":
        rows, cols = np.indices((N, N))
        return (((rows < N // 2) ^ (cols < N // 2))).astype(np.int8)

    if name == "half_split":
        grid = np.zeros((N, N), dtype=np.int8)
        grid[:, N // 2:] = 1
        return grid

    if name == "block_checkerboard":
        if N % block_size:
            raise ValueError("block_size must divide N.")
        rows, cols = np.indices((N, N))
        return (
            ((rows // block_size) + (cols // block_size)) % 2
        ).astype(np.int8)

    raise ValueError(f"Unknown pattern: {name}")


pattern_specs = [
    ("random", {"name": "random"}),
    ("checkerboard", {"name": "checkerboard"}),
    ("vertical_stripes", {"name": "vertical_stripes"}),
    ("quadrant", {"name": "quadrant"}),
    ("half_split", {"name": "half_split"}),
    ("block_checkerboard_k2", {"name": "block_checkerboard", "block_size": 2}),
    ("block_checkerboard_k5", {"name": "block_checkerboard", "block_size": 5}),
]

patterns = {
    label: make_pattern(N=N, **kwargs)
    for label, kwargs in pattern_specs
}

duplicate_pairs = [
    (a, b)
    for a, b in combinations(patterns, 2)
    if np.array_equal(patterns[a], patterns[b])
]
print("Exact duplicate pairs:", duplicate_pairs)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (name, grid) in zip(axes.ravel(), patterns.items()):
    ax.imshow(grid, interpolation="nearest")
    ax.set_title(name.replace("_", " "))
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.ravel()[len(patterns):]:
    ax.axis("off")
fig.suptitle("Canonical 50:50 planting arrangements, N=10")
fig.tight_layout()
plt.show()

analysis_pattern_names = [
    "random",
    "checkerboard",
    "vertical_stripes",
    "quadrant",
    "half_split",
    "block_checkerboard_k2",
]


# 10. Disease incidence and yield

For stochastic replicate \(r\),

\[
I_r(t)
=
\frac1{M}
\sum_{i=1}^{M}\mathbf 1_{\{X_{ir}(t)=I\}}.
\]

For cell \(i\), the empirical final infection probability is

\[
f_i(T)
=
\frac1R
\sum_{r=1}^{R}
\mathbf 1_{\{X_{ir}(T)=I\}}.
\]

The realised field yield is

\[
Y_r(T)
=
\frac1M
\sum_i
\left[
Y_{v_i}^{I}\mathbf 1_{\{X_{ir}(T)=I\}}
+
Y_{v_i}^{H}\mathbf 1_{\{X_{ir}(T)\in\{S,L\}\}}
\right].
\]

Because each cell represents the same planted area, averaging cell-specific cultivar yields gives the field yield in \(\mathrm{t\,ha}^{-1}\).

The primary manuscript scenario remains:

- ten vectors per plant;
- one initially infectious plant;
- initially virus-free vectors;
- no roguing.

For fairness, the same table of random initial infection locations is reused across planting patterns.


In [ ]:
# ---------------------------------------------------------------------
# Primary arrangement experiment
# ---------------------------------------------------------------------
ARTICLE_VECTOR_PRESSURE = 10
ARTICLE_INITIAL_INFECTIOUS = 1

selected_kernel = kernel_cache[float(SELECTED_KERNEL_SCALE)]

# Shared random-number blocks across all planting patterns.
PATTERN_INITIAL_SITE_SEED = MASTER_SEED + 2_000_000
PATTERN_EVENT_SEED = MASTER_SEED + 2_100_000

pattern_results = {}
arrangement_rows = []

pattern_start = time.time()

for pattern_name in analysis_pattern_names:
    result = run_spatial_ensemble(
        patterns[pattern_name],
        selected_kernel,
        vectors_per_plant=ARTICLE_VECTOR_PRESSURE,
        n_initial_infectious=ARTICLE_INITIAL_INFECTIOUS,
        n_runs=N_PATTERN_RUNS,
        initial_site_seed=PATTERN_INITIAL_SITE_SEED,
        event_seed=PATTERN_EVENT_SEED,
        observation_times=OBS_TIMES,
    )

    pattern_results[pattern_name] = result

    yields = result["yields"]
    incidence = result["final_incidence"]

    arrangement_rows.append({
        "pattern": pattern_name,
        "mean_yield": float(np.mean(yields)),
        "yield_sd": float(np.std(yields, ddof=1)) if len(yields) > 1 else np.nan,
        "yield_se": float(np.std(yields, ddof=1) / np.sqrt(len(yields))) if len(yields) > 1 else np.nan,
        "mean_final_incidence": float(np.mean(incidence)),
        "incidence_sd": float(np.std(incidence, ddof=1)) if len(incidence) > 1 else np.nan,
        "incidence_se": float(np.std(incidence, ddof=1) / np.sqrt(len(incidence))) if len(incidence) > 1 else np.nan,
    })

arrangement_summary = pd.DataFrame(arrangement_rows).sort_values(
    "mean_yield", ascending=False
)

print(f"Arrangement runtime: {time.time() - pattern_start:.1f} s")
arrangement_summary

In [ ]:
# Mean epidemic trajectories by planting arrangement.
plt.figure(figsize=(9, 5.5))
for name in analysis_pattern_names:
    mean_i = pattern_results[name]["incidence"].mean(axis=0)
    plt.plot(OBS_TIMES, mean_i, label=name)
plt.xlabel("Time (days)")
plt.ylabel("Field incidence")
plt.title("Mean disease trajectories by planting arrangement")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Mean viruliferous-vector prevalence.
plt.figure(figsize=(9, 5.5))
for name in analysis_pattern_names:
    mean_v = pattern_results[name]["vector_prevalence"].mean(axis=0)
    plt.plot(OBS_TIMES, mean_v, label=name)
plt.xlabel("Time (days)")
plt.ylabel("Viruliferous-vector prevalence")
plt.title("Mean vector infection trajectories by planting arrangement")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Final infection-probability maps.
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, name in zip(axes.ravel(), analysis_pattern_names):
    image = ax.imshow(
        pattern_results[name]["infection_probability"],
        vmin=0,
        vmax=1,
        interpolation="nearest",
    )
    ax.set_title(name.replace("_", " "))
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.8, label="P(infectious at T)")
fig.suptitle("Final spatial infection probability")
plt.show()


# 11. Does the planting conclusion survive uncertainty in \(a\)?

A single operational scale is needed for the principal analysis, but the current mean-field procedure does not prove that \(a\) is biologically known.

Therefore, the arrangement analysis is repeated at:

- the lower edge of the one-SE acceptable scale region;
- the selected operational scale;
- the upper edge of the one-SE acceptable scale region.

If the acceptable region collapses to one point, the set contains one value.

For each scale, patterns are compared with common initial infection locations and common event-seed blocks.

Useful robustness diagnostics are:

1. the highest-yielding arrangement at each \(a\);
2. Spearman rank correlation of pattern yields with the ranking at the selected \(a\);
3. whether yield differences are larger than their Monte Carlo uncertainty.

A strong paper-level conclusion should not depend on an arbitrary point inside a weakly identified kernel-scale plateau.


In [ ]:
A_ROBUSTNESS_SET = np.unique(np.array([
    selection_all["acceptable_a_min"],
    selection_all["selected_a"],
    selection_all["acceptable_a_max"],
], dtype=float))

kernel_robustness_rows = []

for a_index, a in enumerate(A_ROBUSTNESS_SET):
    kernel = kernel_cache[float(a)]

    for pattern_name in analysis_pattern_names:
        result = run_spatial_ensemble(
            patterns[pattern_name],
            kernel,
            vectors_per_plant=ARTICLE_VECTOR_PRESSURE,
            n_initial_infectious=ARTICLE_INITIAL_INFECTIOUS,
            n_runs=N_KERNEL_ROBUSTNESS_RUNS,
            initial_site_seed=MASTER_SEED + 3_000_000,
            event_seed=MASTER_SEED + 3_100_000,
            observation_times=OBS_TIMES,
        )

        kernel_robustness_rows.append({
            "a": float(a),
            "pattern": pattern_name,
            "mean_yield": float(result["yields"].mean()),
            "mean_final_incidence": float(result["final_incidence"].mean()),
        })

kernel_robustness = pd.DataFrame(kernel_robustness_rows)

# Ranking diagnostics relative to the selected a.
reference = (
    kernel_robustness[
        np.isclose(kernel_robustness["a"], SELECTED_KERNEL_SCALE)
    ]
    .set_index("pattern")["mean_yield"]
)

ranking_rows = []
for a, group in kernel_robustness.groupby("a"):
    values = group.set_index("pattern")["mean_yield"].reindex(reference.index)
    rho = spearmanr(reference.values, values.values).statistic
    best_pattern = values.idxmax()
    ranking_rows.append({
        "a": float(a),
        "best_pattern": best_pattern,
        "Spearman_vs_selected_a": float(rho),
    })

kernel_ranking_robustness = pd.DataFrame(ranking_rows)

display(kernel_robustness)
display(kernel_ranking_robustness)


# 12. Spatial arrangement metrics: audited definitions

The draft manuscript listed a large family of spatial indices. Some names, especially bespoke uses of EMD, Morisita–Horn, and “ICM”, require a precise reference distribution or canonical landscape definition before they can be interpreted scientifically.

This notebook therefore computes a reproducible core of explicitly defined descriptors:

1. **Heterospecific-neighbour index (HNI)** using Moore neighbours;
2. **unlike rook-join fraction**;
3. **local Shannon entropy** in a \(3\times3\) window;
4. **local Simpson diversity**;
5. **boundary length per cell**;
6. **patch density** under Moore connectivity;
7. **mean patch area**;
8. **largest-patch fraction**;
9. **same-variety Moore connectivity**;
10. **Moran's \(I\)** for the binary cultivar map.

Additional metrics should be added only with explicit formulas, edge corrections, and validated reference implementations. The notebook does not preserve a metric name merely to reproduce a previous numerical result.


In [ ]:
def neighbours(index, n_rows, n_cols, moore=True):
    row, col = divmod(index, n_cols)
    offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if moore:
        offsets += [(-1, -1), (-1, 1), (1, -1), (1, 1)]

    result = []
    for dr, dc in offsets:
        rr, cc = row + dr, col + dc
        if 0 <= rr < n_rows and 0 <= cc < n_cols:
            result.append(rr * n_cols + cc)
    return result


def patch_areas(grid):
    n_rows, n_cols = grid.shape
    flat = grid.ravel()
    visited = np.zeros(flat.size, dtype=bool)
    areas = []

    for start in range(flat.size):
        if visited[start]:
            continue

        target = flat[start]
        queue = deque([start])
        visited[start] = True
        area = 0

        while queue:
            current = queue.popleft()
            area += 1

            for nxt in neighbours(current, n_rows, n_cols, moore=True):
                if not visited[nxt] and flat[nxt] == target:
                    visited[nxt] = True
                    queue.append(nxt)

        areas.append(area)

    return np.asarray(areas, dtype=float)


def arrangement_metrics(grid, local_window=3):
    n_rows, n_cols = grid.shape
    flat = grid.ravel().astype(float)
    n_cells = flat.size

    heterospecific = []
    same_type = []

    for i in range(n_cells):
        neigh = neighbours(i, n_rows, n_cols, moore=True)
        diff = np.mean([flat[j] != flat[i] for j in neigh])
        heterospecific.append(diff)
        same_type.append(1.0 - diff)

    unlike_joins = 0
    total_rook_joins = 0

    for r in range(n_rows):
        for c in range(n_cols):
            if c + 1 < n_cols:
                total_rook_joins += 1
                unlike_joins += grid[r, c] != grid[r, c + 1]
            if r + 1 < n_rows:
                total_rook_joins += 1
                unlike_joins += grid[r, c] != grid[r + 1, c]

    radius = local_window // 2
    shannon = []
    simpson = []

    for r in range(n_rows):
        for c in range(n_cols):
            window = grid[
                max(0, r - radius):min(n_rows, r + radius + 1),
                max(0, c - radius):min(n_cols, c + radius + 1),
            ].ravel()

            p = np.array([
                np.mean(window == 0),
                np.mean(window == 1),
            ])
            p_nonzero = p[p > 0]
            shannon.append(-np.sum(p_nonzero * np.log(p_nonzero)))
            simpson.append(1.0 - np.sum(p * p))

    areas = patch_areas(grid)

    centred = flat - flat.mean()
    denominator = np.sum(centred ** 2)
    numerator = 0.0
    weight_sum = 0.0

    for i in range(n_cells):
        for j in neighbours(i, n_rows, n_cols, moore=False):
            numerator += centred[i] * centred[j]
            weight_sum += 1.0

    moran_i = (
        (n_cells / weight_sum) * numerator / denominator
        if denominator > 0 and weight_sum > 0
        else np.nan
    )

    return {
        "HNI_Moore": float(np.mean(heterospecific)),
        "unlike_rook_join_fraction": float(unlike_joins / total_rook_joins),
        "local_Shannon_3x3": float(np.mean(shannon)),
        "local_Simpson_3x3": float(np.mean(simpson)),
        "boundary_per_cell": float(unlike_joins / n_cells),
        "patch_density_Moore": float(len(areas) / n_cells),
        "mean_patch_area": float(np.mean(areas)),
        "largest_patch_fraction": float(np.max(areas) / n_cells),
        "same_type_connectivity_Moore": float(np.mean(same_type)),
        "Moran_I_rook": float(moran_i),
    }


metric_rows = []
for name in analysis_pattern_names:
    row = {"pattern": name}
    row.update(arrangement_metrics(patterns[name]))
    metric_rows.append(row)

metric_table = pd.DataFrame(metric_rows)
metric_table


# 13. Exploratory relationship between spatial metrics and outcomes

With only six non-duplicate canonical planting patterns, model selection among pairs of spatial metrics is necessarily exploratory.

For a metric pair \((X_1,X_2)\),

\[
Y=b_0+b_1X_1+b_2X_2.
\]

Leave-one-pattern-out prediction gives

\[
R^2_{\mathrm{LOPO}}
=
1-
\frac{\sum_p(Y_p-\widehat Y_{-p})^2}
{\sum_p(Y_p-\bar Y)^2}.
\]

Negative values are retained because they honestly indicate predictions worse than the intercept-only mean.

Bootstrap Spearman correlation is reported only as a descriptive ranking-stability diagnostic. It is not independent evidence of a general law.

Any final manuscript claim about “the best metric pair” should be revisited using a much larger ensemble of distinct planting maps, not merely the six canonical designs.


In [ ]:
def lopo_r2(X, y):
    predictions = np.empty_like(y, dtype=float)

    for held_out in range(len(y)):
        train = np.arange(len(y)) != held_out
        model = LinearRegression().fit(X[train], y[train])
        predictions[held_out] = model.predict(X[[held_out]])[0]

    denominator = np.sum((y - y.mean()) ** 2)
    if denominator <= 0:
        return np.nan, predictions

    score = 1.0 - np.sum((y - predictions) ** 2) / denominator
    return float(score), predictions


def bootstrap_rank_stability(
    X,
    y,
    n_boot=N_METRIC_BOOTSTRAP,
    seed=MASTER_SEED,
):
    rng = np.random.default_rng(seed)
    correlations = []
    n = len(y)

    for _ in range(n_boot):
        sample = rng.integers(0, n, size=n)

        if np.unique(sample).size < 3:
            continue

        model = LinearRegression().fit(X[sample], y[sample])
        prediction = model.predict(X[sample])
        rho = spearmanr(prediction, y[sample]).statistic

        if np.isfinite(rho):
            correlations.append(rho)

    if not correlations:
        return np.nan, np.nan

    correlations = np.asarray(correlations)
    return (
        float(np.median(correlations)),
        float(np.std(correlations, ddof=1)),
    )


analysis_table = metric_table.merge(
    arrangement_summary,
    on="pattern",
    how="inner",
)

metric_names = [c for c in metric_table.columns if c != "pattern"]

pair_rows = []

for response in ("mean_yield", "mean_final_incidence"):
    y = analysis_table[response].to_numpy(float)

    for first, second in combinations(metric_names, 2):
        X = analysis_table[[first, second]].to_numpy(float)

        score, prediction = lopo_r2(X, y)
        median_rho, sd_rho = bootstrap_rank_stability(X, y)

        pair_rows.append({
            "response": response,
            "metric_1": first,
            "metric_2": second,
            "LOPO_R2": score,
            "bootstrap_median_Spearman": median_rho,
            "bootstrap_sd_Spearman": sd_rho,
        })

metric_pair_diagnostics = pd.DataFrame(pair_rows)

metric_pair_diagnostics.sort_values(
    ["response", "LOPO_R2"],
    ascending=[True, False],
).groupby("response").head(5)


# 14. Interpretation framework for the manuscript

## 14.1 What the notebook can establish

If the numerical diagnostics support it, the analysis can establish that:

- vector dispersal is represented by an explicitly defined conservative exchange generator;
- every tagged vector has the intended individual dispersal rate \(\sigma\);
- local vector burden is exactly preserved;
- vector movement has no cultivar-attractiveness term;
- the scale \(a\) is selected from a stated, reproducible multi-context mean-field-consistency criterion;
- uncertainty/weak identifiability in \(a\) is visible rather than hidden;
- planting arrangements are compared under the same movement process and epidemiological parameters;
- arrangement conclusions are stress-tested across the mean-field-compatible range of \(a\).

## 14.2 What the notebook must not claim

The analysis must not claim that:

- mean-field matching directly measures biological whitefly dispersal;
- a context-specific optimum \(a_s^*\) is a different biological dispersal parameter;
- one terminal yield value is sufficient to identify movement;
- \(a\) equals the mean movement distance;
- the \(10\times10\) prototype has the same initial prevalence as the \(10{,}000\)-plant PLOS field;
- six canonical patterns are sufficient to establish a universal two-metric law.

## 14.3 Publication-scale requirements

Before publication-level numerical claims:

1. repeat at \(100\times100\) plants so that one initially infectious plant corresponds to \(10^{-4}\) prevalence, as in the PLOS field;
2. increase Monte Carlo replication until confidence intervals and arrangement ranks stabilise;
3. repeat calibration and arrangement rankings over multiple master seeds;
4. examine edge/boundary assumptions;
5. compare the exponential kernel with empirical alternatives if movement data permit;
6. propagate EpiPvr posterior parameter uncertainty rather than using only point estimates;
7. expand the planting-map ensemble beyond six canonical patterns before making general claims about landscape metrics.

The \(10\times10\) notebook remains a rigorous prototype and methodological test bed.


## 14.4 Total uncertainty is larger than stochastic replicate variability

For outcome \(Y\),

\[
\operatorname{Var}(Y\mid D)
=
E_{\Theta\mid D}
\left[\operatorname{Var}(Y\mid\Theta)\right]
+
\operatorname{Var}_{\Theta\mid D}
\left[E(Y\mid\Theta)\right].
\]

The first term is epidemic process variability; the second is uncertainty propagated from EpiPvr.

The current point-estimate analysis represents the first term only.

## 14.5 Bayesian definition of optimal planting design

With fixed point estimates,

\[
z^*(\widehat\Theta)
=
\arg\max_z E_{\mathrm{CTMC}}[Y\mid z,\widehat\Theta].
\]

With full EpiPvr posterior uncertainty,

\[
\boxed{
z^*
=
\arg\max_z
E_{\Theta\mid D}
\left[
E_{\mathrm{CTMC}}(Y\mid z,\Theta)
\right].
}
\]

Cropmix should eventually report probabilities such as

\[
P(Y(z_1)>Y(z_2)\mid D)
\]

and the posterior frequency with which each candidate design is optimal.



# 15. Executable reproducibility checklist

The following checks are mandatory:

### Source/parameter audit
- CBSD acquisition, inoculation, clearance, mortality, dispersal and season parameters equal the audited PLOS implementation values.

### Kernel audit
- exactly 100 values in `SENSITIVITY_GRID`;
- first value \(0.05\), last value \(100\);
- zero diagonal;
- row sums equal one;
- symmetry;
- unordered pair mass \(M/2\);
- tagged movement rate exactly \(\sigma\).

### Stochastic-state audit
- valid plant states only;
- non-negative vector counts;
- \(V_i^A+V_i^B\le m\);
- hence \(H_i=m-V_i^A-V_i^B\ge0\);
- deterministic replay under identical initial-site and event seeds;
- no event after \(T\) changes the harvest state.

### Design audit
- exact 50:50 composition;
- duplicate planting maps detected and excluded from inferential analysis.

### Calibration audit
- one common selected \(a\);
- context-specific minima retained only as diagnostics;
- boundary minima flagged;
- acceptable interval recorded;
- leave-one-context-out sensitivity recorded;
- arrangement ranking checked over the acceptable scale region.

### Computational provenance
- Python/platform/package versions;
- all master seeds and replication counts;
- run manifest;
- CSV outputs;
- SHA-256 checksums.


### Inferential-hierarchy audit
- Bayesian EpiPvr inference is labelled as parameter uncertainty.
- Branching-process epidemic probability is labelled as an early-invasion quantity.
- No branching-process/Gillespie equality is claimed without harmonised assumptions.
- The current SPT engine is not silently used for PT transmission.
- Posterior samples, when supplied, are coherent joint rows.
- Hour/day conversion is explicit.
- Point-estimate results are labelled as conditional on \(\widehat\Theta\).


In [ ]:
# ---------------------------------------------------------------------
# Executable reproducibility checks
# ---------------------------------------------------------------------
checks = []

def check(name, condition, detail):
    passed = bool(condition)
    checks.append({
        "check": name,
        "passed": passed,
        "detail": str(detail),
    })
    if not passed:
        raise AssertionError(f"{name}: {detail}")


# Parameter audit
expected = {
    "SUSC_acquisition": 15.31,
    "SUSC_inoculation": 1.34,
    "RES_acquisition": 4.84,
    "RES_inoculation": 0.42,
    "gamma": 1.0 / 30.0,
    "sigma": 0.45,
    "omega": 0.19,
    "clearance": 19.37,
    "T": 360.0,
}
observed = {
    "SUSC_acquisition": SUSC.acquisition,
    "SUSC_inoculation": SUSC.inoculation,
    "RES_acquisition": RES.acquisition,
    "RES_inoculation": RES.inoculation,
    "gamma": SUSC.latency_progression,
    "sigma": SIGMA,
    "omega": OMEGA,
    "clearance": VECTOR_CLEARANCE,
    "T": T_END,
}
check(
    "Audited CBSD parameter values",
    all(np.isclose(observed[k], v) for k, v in expected.items()),
    observed,
)

check(
    "Point transmission draw matches audited values",
    np.isclose(POINT_TRANSMISSION_DRAW.susc_acquisition, SUSC.acquisition)
    and np.isclose(POINT_TRANSMISSION_DRAW.susc_inoculation, SUSC.inoculation)
    and np.isclose(POINT_TRANSMISSION_DRAW.res_acquisition, RES.acquisition)
    and np.isclose(POINT_TRANSMISSION_DRAW.res_inoculation, RES.inoculation)
    and np.isclose(POINT_TRANSMISSION_DRAW.vector_clearance, VECTOR_CLEARANCE),
    POINT_TRANSMISSION_DRAW,
)

# Requested calibration grid.
check(
    "Sensitivity grid has 100 values",
    len(SENSITIVITY_GRID) == 100,
    len(SENSITIVITY_GRID),
)
check(
    "Sensitivity grid starts at 0.05",
    np.isclose(SENSITIVITY_GRID[0], 0.05),
    SENSITIVITY_GRID[0],
)
check(
    "Sensitivity grid ends at 100",
    np.isclose(SENSITIVITY_GRID[-1], 100.0),
    SENSITIVITY_GRID[-1],
)

# Movement-matrix audit.
for a in (0.05, 1.0, 100.0):
    k = kernel_cache[float(SENSITIVITY_GRID[
        np.argmin(np.abs(SENSITIVITY_GRID - a))
    ])] if a not in kernel_cache else kernel_cache[a]

    P = k["probabilities"]
    check(
        f"Kernel diagonal zero at a≈{a}",
        np.allclose(np.diag(P), 0.0),
        np.max(np.abs(np.diag(P))),
    )
    check(
        f"Kernel rows sum to one at a≈{a}",
        np.allclose(P.sum(axis=1), 1.0, atol=1e-10),
        np.max(np.abs(P.sum(axis=1) - 1.0)),
    )
    check(
        f"Kernel symmetric at a≈{a}",
        np.allclose(P, P.T, atol=1e-10),
        np.max(np.abs(P - P.T)),
    )
    check(
        f"Pair mass M/2 at a≈{a}",
        np.isclose(np.triu(P, 1).sum(), N * N / 2.0, atol=1e-9),
        np.triu(P, 1).sum(),
    )
    check(
        f"Tagged vector dispersal rate sigma at a≈{a}",
        np.allclose(SIGMA * P.sum(axis=1), SIGMA, atol=1e-10),
        (SIGMA * P.sum(axis=1)).min(),
    )

# Pattern composition and uniqueness.
for name, grid in patterns.items():
    check(
        f"Pattern {name} is 50:50",
        np.sum(grid == 0) == np.sum(grid == 1) == grid.size // 2,
        (np.sum(grid == 0), np.sum(grid == 1)),
    )

check(
    "Inferential pattern list has no duplicates",
    len({patterns[name].tobytes() for name in analysis_pattern_names})
    == len(analysis_pattern_names),
    analysis_pattern_names,
)

# Deterministic replay.
audit_kernel = prepare_kernel(4, 4, 1.0)
audit_grid = np.zeros((4, 4), dtype=np.int8)

replay_1 = run_spatial_ensemble(
    audit_grid,
    audit_kernel,
    vectors_per_plant=2,
    n_initial_infectious=1,
    n_runs=3,
    initial_site_seed=MASTER_SEED + 8_000_000,
    event_seed=MASTER_SEED + 8_100_000,
    observation_times=np.linspace(0.0, 20.0, 5),
    t_end=20.0,
)
replay_2 = run_spatial_ensemble(
    audit_grid,
    audit_kernel,
    vectors_per_plant=2,
    n_initial_infectious=1,
    n_runs=3,
    initial_site_seed=MASTER_SEED + 8_000_000,
    event_seed=MASTER_SEED + 8_100_000,
    observation_times=np.linspace(0.0, 20.0, 5),
    t_end=20.0,
)

check(
    "Deterministic replay",
    np.array_equal(replay_1["final_states"], replay_2["final_states"])
    and np.array_equal(replay_1["yields"], replay_2["yields"])
    and np.array_equal(replay_1["incidence"], replay_2["incidence"]),
    "Identical seed blocks reproduce identical outputs",
)

# Calibration diagnostics.
check(
    "Single all-context selected a exists",
    np.isfinite(SELECTED_KERNEL_SCALE) and SELECTED_KERNEL_SCALE > 0,
    SELECTED_KERNEL_SCALE,
)
check(
    "Selected a lies inside requested domain",
    0.05 <= SELECTED_KERNEL_SCALE <= 100.0,
    SELECTED_KERNEL_SCALE,
)

# Boundary optimum is a diagnostic rather than a hard error.
checks.append({
    "check": "Literal all-context optimum at scan boundary (diagnostic)",
    "passed": True,
    "detail": selection_all["boundary_warning"],
})

checklist_results = pd.DataFrame(checks)
print(f"{checklist_results['passed'].sum()}/{len(checklist_results)} mandatory checks passed.")
checklist_results

In [ ]:
# ---------------------------------------------------------------------
# Save numerical outputs and computational provenance
# ---------------------------------------------------------------------
OUTPUT_DIR = Path("spatial_model_outputs_robust")
REPRO_DIR = OUTPUT_DIR / "reproducibility"
OUTPUT_DIR.mkdir(exist_ok=True)
REPRO_DIR.mkdir(exist_ok=True)

parameter_table.to_csv(OUTPUT_DIR / "parameters.csv", index=False)
meanfield_benchmarks.to_csv(OUTPUT_DIR / "meanfield_benchmarks.csv", index=False)
kernel_geometry.to_csv(OUTPUT_DIR / "kernel_geometry.csv", index=False)
movement_audit.to_csv(OUTPUT_DIR / "movement_audit.csv", index=False)
CALIBRATION_CONTEXTS.to_csv(OUTPUT_DIR / "calibration_contexts.csv", index=False)
calibration_long.to_csv(OUTPUT_DIR / "kernel_calibration_long.csv", index=False)
joint_all.to_csv(OUTPUT_DIR / "kernel_joint_all_contexts.csv", index=False)
joint_core.to_csv(OUTPUT_DIR / "kernel_joint_plos_core.csv", index=False)
selection_summary.to_csv(OUTPUT_DIR / "kernel_selection_summary.csv", index=False)
a_summary_table.to_csv(OUTPUT_DIR / "kernel_a_summary.csv", index=False)
context_optima.to_csv(OUTPUT_DIR / "kernel_context_optima_diagnostic.csv", index=False)
loco_table.to_csv(OUTPUT_DIR / "kernel_leave_one_context_out.csv", index=False)
arrangement_summary.to_csv(OUTPUT_DIR / "arrangement_summary.csv", index=False)
kernel_robustness.to_csv(OUTPUT_DIR / "arrangement_kernel_robustness.csv", index=False)
kernel_ranking_robustness.to_csv(OUTPUT_DIR / "arrangement_ranking_robustness.csv", index=False)
metric_table.to_csv(OUTPUT_DIR / "arrangement_metrics.csv", index=False)
metric_pair_diagnostics.to_csv(OUTPUT_DIR / "metric_pair_diagnostics.csv", index=False)
checklist_results.to_csv(REPRO_DIR / "checklist_results.csv", index=False)

package_names = [
    "numpy",
    "pandas",
    "matplotlib",
    "scipy",
    "scikit-learn",
    "numba",
    "nbformat",
]

package_versions = {}
for package in package_names:
    try:
        package_versions[package] = importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        package_versions[package] = "not installed"

manifest = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
    "packages": package_versions,
    "master_seed": MASTER_SEED,
    "fast_mode": FAST_MODE,
    "replications": {
        "calibration_per_a_context": N_CALIBRATION_RUNS,
        "pattern_primary": N_PATTERN_RUNS,
        "pattern_kernel_robustness": N_KERNEL_ROBUSTNESS_RUNS,
        "calibration_bootstrap": N_BOOTSTRAP,
        "metric_bootstrap": N_METRIC_BOOTSTRAP,
    },
    "field": {
        "rows": N,
        "columns": N,
        "plants": N * N,
        "plant_spacing_m": PLANT_SPACING_M,
    },
    "movement": {
        "kernel": "balanced exponential",
        "formula": "exp(-d/a)",
        "pair_generator": "lambda_ij = m * sigma * P_ij",
        "sigma": SIGMA,
        "selected_a": SELECTED_KERNEL_SCALE,
        "selected_mean_step_distance": SELECTED_MEAN_STEP_DISTANCE,
        "acceptable_a_min": selection_all["acceptable_a_min"],
        "acceptable_a_max": selection_all["acceptable_a_max"],
        "sensitivity_grid_n": len(SENSITIVITY_GRID),
        "sensitivity_grid_min": float(SENSITIVITY_GRID.min()),
        "sensitivity_grid_max": float(SENSITIVITY_GRID.max()),
    },
    "parameters": observed,
}

with open(REPRO_DIR / "run_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

# Seed ledger
seed_ledger = pd.DataFrame([
    ["calibration initial-site base", MASTER_SEED + 10_000],
    ["calibration event base", MASTER_SEED + 100_000],
    ["calibration bootstrap base", MASTER_SEED + 1_000_000],
    ["pattern initial sites", PATTERN_INITIAL_SITE_SEED],
    ["pattern event stream", PATTERN_EVENT_SEED],
    ["kernel robustness initial sites", MASTER_SEED + 3_000_000],
    ["kernel robustness event stream", MASTER_SEED + 3_100_000],
    ["replay initial sites", MASTER_SEED + 8_000_000],
    ["replay event stream", MASTER_SEED + 8_100_000],
], columns=["purpose", "seed_or_seed_block_start"])
seed_ledger.to_csv(REPRO_DIR / "seed_ledger.csv", index=False)

manifest

In [ ]:
# SHA-256 checksums for all generated outputs.
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


files_to_hash = sorted(
    p for p in OUTPUT_DIR.rglob("*")
    if p.is_file() and p.name != "SHA256SUMS.csv"
)

checksum_table = pd.DataFrame([
    {
        "file": str(p.relative_to(OUTPUT_DIR)),
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    }
    for p in files_to_hash
])

checksum_table.to_csv(REPRO_DIR / "SHA256SUMS.csv", index=False)
checksum_table

# 16. Reproducible reporting template

## 16.1 Parameter-inference statement

> Transmission parameters were inherited from the EpiPvr/PLOS modelling lineage. EpiPvr estimates acquisition, inoculation and vector-clearance rates from access-period experiments using Bayesian models fitted by MCMC in Stan. The primary spatial analysis used published point summaries of those rates and therefore conditions on a fixed parameter vector. Stochastic variation among spatial simulations represents epidemic process variability and does not include posterior uncertainty in transmission parameters.

If posterior propagation is later used:

> Joint posterior draws from EpiPvr were propagated through the spatial CTMC using nested Monte Carlo, integrating both transmission-parameter uncertainty and stochastic epidemic variability.

## 16.2 Branching-process statement

> EpiPvr also uses estimated transmission parameters in a multitype branching-process field model to infer epidemic establishment probabilities following rare inoculum introduction. This invasion calculation is complementary to the spatial Gillespie model. The published EpiPvr branching process treats space implicitly and includes local demographic/removal assumptions that are not identical to those of the present no-roguing, terminal-harvest arrangement experiment. We therefore do not interpret EpiPvr establishment probability and the present finite-season spatial outcomes as numerically interchangeable without an explicitly harmonised validation scenario.

## 16.3 Spatial movement and kernel statement

> Vector movement between plant locations was represented by a symmetric exponentially distance-weighted pair-exchange process. For unordered plant pair \(\{i,j\}\), exchanges occurred at rate \(\lambda_{ij}=m\sigma P_{ij}\), where \(P\) is a symmetric row-stochastic balancing of \(e^{-d_{ij}/a}\). This construction ensures that each individual vector disperses at rate \(\sigma\) while preserving exactly \(m\) vectors at every plant.

> The spatial scale \(a\) was not interpreted as directly measured vector dispersal. Instead, a common operational value was selected by minimising discrepancy between spatial ensemble trajectories and the corresponding finite-size PLOS mean-field model across multiple epidemiological contexts. Plant incidence and viruliferous-vector prevalence trajectories were used for calibration, while final yield was retained as an external validation outcome.

Always report both \(a=a_{\mathrm{MF}}\) and the finite-field mean movement distance \(\bar d(a_{\mathrm{MF}})\).

## 16.4 Scientific checklist before manuscript use

- [ ] Run with `FAST_MODE=False`.
- [ ] Confirm Monte Carlo standard errors are acceptably small.
- [ ] Inspect context-specific dynamic-score curves.
- [ ] Inspect the all-context one-SE interval.
- [ ] Inspect leave-one-context-out selected scales.
- [ ] Check whether the literal optimum is on a scan boundary.
- [ ] Examine incidence and vector-prevalence residuals separately.
- [ ] Confirm final yield agrees reasonably with the mean-field reference.
- [ ] Confirm arrangement rankings are stable across the acceptable \(a\) interval.
- [ ] Repeat with multiple master seeds.
- [ ] Repeat at \(100\times100\) before equating initial prevalence with the published \(10^{-4}\).
- [ ] Propagate EpiPvr posterior uncertainty in a later uncertainty analysis.
- [ ] For branching-process/Gillespie validation, harmonise all local assumptions before numerical comparison.
- [ ] Validate additional landscape metrics against canonical definitions.

The notebook is a scientific model specification, not merely a plotting script.


# 17. Core references and final conceptual summary

1. Tankam Chedjou, I., Donnelly, R. & Gilligan, C. A. (2025). *Optimizing crop varietal mixtures for viral disease management: A case study on cassava virus epidemics.* PLOS Computational Biology 21(9): e1012842. https://doi.org/10.1371/journal.pcbi.1012842

2. Donnelly, R., Tankam Chedjou, I. & Gilligan, C. A. (2026). *Plant pathogen profiling with the EpiPvr package.* Methods in Ecology and Evolution 17: 837–849. https://doi.org/10.1111/2041-210x.70219

3. EpiPvr R package: https://cran.r-project.org/package=EpiPvr

4. PLOS mixture-model implementation: https://github.com/israeltankam/mixture-simulator

5. Gillespie, D. T. (1976). A general method for numerically simulating the stochastic time evolution of coupled chemical reactions. *Journal of Computational Physics* 22: 403–434.

6. Gillespie, D. T. (1977). Exact stochastic simulation of coupled chemical reactions. *Journal of Physical Chemistry* 81: 2340–2361.

---

The full chain is

\[
\boxed{
D
\rightarrow
p(\Theta\mid D)
\rightarrow
\begin{cases}
P_{\mathrm{establishment}} & \text{branching process},\\
\bar X(t) & \text{mean-field ODE},\\
X(t) & \text{spatial Gillespie CTMC}.
\end{cases}
}
\]

For future Cropmix,

\[
\boxed{
p(\mathcal O\mid D,z)
=
\int
p_{\mathrm{CTMC}}(\mathcal O\mid z,\Theta)
p(\Theta\mid D)\,d\Theta.
}
\]

That equation is the mathematical bridge between EpiPvr and Cropmix.
